<a href="https://colab.research.google.com/github/hanjiadong0/chatbot-/blob/RL/rl_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [218]:
from google.colab import drive
drive.mount('/content/drive')
import sys
import os


# Specify the path to the directory containing usage_logger.py in your Google Drive
drive_module_path = '/content/drive/My Drive/thesis_assistant/thesis_assistant_modular'

# Add the directory to the Python path
if drive_module_path not in sys.path:
    sys.path.append(drive_module_path)

try:
    # Import the usage_logger module
    import usage_logger
    print("Successfully imported usage_logger.")
except ImportError:
    print(f"Error: Could not import usage_logger. Make sure 'usage_logger.py' is in '{drive_module_path}'")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully imported usage_logger.



### RL Optimizer
This thesis simulates a reinforcement learning framework for thesis-writing assistance, combining behavior modeling, ethical oversight, advisor interaction dynamics, and noisy writing quality approximations. While true real-world RLHF reward models remain out of reach at this stage, this controlled simulation provides a sandbox to explore adaptive policy learning for human-in-the-loop academic coaching.

## Proposed RL Agent State Structure

This outlines a potential structure for the State that the Reinforcement Learning agent (overseeing project evolution and ethics) would observe. This state combines information from the Ethics Module with broader thesis progress details.


The state would likely be represented as a numerical vector or a structured object that the RL model can process.

**Components of the State:**

1.  **Ethical State Features (from Ethics Module):**
   
2.  **Thesis Progress Features:**
    *   **Current Thesis Stage:** A categorical or numerical representation of the current stage of the thesis (e.g., 0 for planning, 1 for literature review, 2 for methodology, 3 for writing, etc.).
    *   **Task Completion:** Percentage of planned tasks completed for the current stage or overall project.
    *   **Time-based Metrics:** Time spent on the project recently, time remaining until deadlines.
    *   **Advisor Feedback Status:** A flag or metric indicating the presence and recency of unaddressed advisor feedback.

3.  **Performance Features:**
    *   **Work Quality Score:** A metric representing the quality of recent thesis work (this would be challenging to define and might require human evaluation or proxy metrics).
    *   **Progress Rate:** A measure of how quickly tasks are being completed or milestones are being reached.

**Combining the State:**

These individual features would be combined into a single state representation that the RL agent's model can process. For a neural network-based RL model, this would typically be a flattened numerical vector. Categorical features would need to be appropriately encoded (e.g., one-hot encoding).

**Next Steps for Implementation (for later):**

*   Define the specific numerical or categorical representation for each state feature.
*   Develop the logic within the thesis assistant to collect and compile this information into the state vector at each time step.
*   Ensure the Ethics Module submodules (Usage_Logger, AI_Detector, etc.) are providing the necessary data points in a format that can be easily integrated into the state.

In [219]:
!pip install streamlit gymnasium stable-baselines3
!pip install numpy # Ensure numpy is installed if not already
!pip install pandas # Ensure pandas is installed if not already
!pip install scipy # Ensure scipy is installed if not already
!pip install langgraph
# --- START OF FILE: langgraph_policy.py ---

from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, END
import numpy as np
import json
import os
import gymnasium as gym
import streamlit as st
import numpy as np
import random
from stable_baselines3 import PPO


### Part 0 Mock Modules:


In [232]:
# PART 0: Mock Modules and State Definition
# ===========================================================
#
# These are simplified mock implementations of the different
# components (like Ethics, Writing, Emotion, Idea Generation)
# of the Thesis Assistant.
#
# - Each module manages its own state variables.
# - They can potentially interact with a shared log.
# - Their state variables contribute to the overall environment state.
# - Their behavior can be influenced by the loaded configuration.
#
# -----------------------------------------------------------
# ✅ WHY MOCK MODULES?
# -----------------------------------------------------------
# - Allows development and testing of the RL environment and
#   policy without needing fully functional assistant components.
# - Provides a dynamic state that the RL agent can observe and influence.
# - Facilitates testing of config-driven environment dynamics.
# ===========================================================

import numpy as np
import random
import gymnasium as gym # Use gymnasium instead of gym for modern API
from typing import Dict, List, Any, TypedDict # Import necessary types
from pydantic import BaseModel # Import BaseModel for type checking if needed


# Redefine ThesisState if not already defined in a previous cell that is guaranteed to run before this one.
# Assuming it is defined in a previous cell (e.g., V04e15e7123d or M4UvshTCIBv3).
# If not, uncomment and define it here:
# class CoreState(TypedDict):
#     stage: str
#     advisor_trust: float
#     creativity_score: float
#     ethical_flags: float
#     ai_usage: float
#     thesis_quality: float
#     deadline_ratio: float
#     thesis_difficulty: float
#     student_autonomy: float
#     language_proficiency: float
#     emotional_state: float
#     timestep: int

# class ThesisState(TypedDict):
#     core: Dict[str, float]
#     policy_trace: List[str]
#     log: List[str]
#     config: Dict[str, Any]
#     env: Any
#     done: bool
#     truncated: bool


class SimpleMockEthicsModule:
    """A simple mock module simulating ethical considerations."""
    def __init__(self, config: Dict[str, Any]):
        # Access config parameters based on RLConfig schema keys
        self.config = config
        self.ethical_flags: float = 0.0 # State variable: Aggregated ethical concerns (0.0 to 1.0)
        self.ai_usage: float = 0.0      # State variable: Level of AI assistance used (0.0 to 1.0)
        # Access reward config items using the correct key 'reward_config'
        # Access the Pydantic RewardItem instance and its 'value' attribute
        reward_config = self.config.get("reward_config", {})
        ethical_boundary_crossed_reward_item = reward_config.get("ethical_boundary_crossed")
        self._ethical_penalty_value = getattr(ethical_boundary_crossed_reward_item, "value", -5.0) if ethical_boundary_crossed_reward_item else -5.0

        # Access other config parameters as needed, using get with defaults
        self._ethics_threshold = self.config.get("ethics_threshold", 0.7) # Example: configurable threshold


    def update(self, action: str, log: List[str]):
        """Simulate state update based on action and log."""
        # Example: Action 'eth_3' increases ethical flags
        if action == "eth_3":
            self.ethical_flags = np.clip(self.ethical_flags + 0.2, 0.0, 1.0)
            log.append("Ethics: Academic concern logged.")
        # Example: Action 'eth_1' decreases AI usage
        elif action == "eth_1":
             self.ai_usage = np.clip(self.ai_usage - 0.2, 0.0, 1.0)
             log.append("Ethics: AI restriction proposed.")
        # Example: Ethical flags naturally decay over time
        self.ethical_flags = np.clip(self.ethical_flags * 0.98, 0.0, 1.0) # Decay

        # Simulate a random event based on state
        if self.ethical_flags > self._ethics_threshold and random.random() < 0.1:
             log.append("Ethics: High ethical flags triggered a warning.")


    def reset(self):
        """Reset module state for a new episode."""
        self.ethical_flags = 0.0
        self.ai_usage = 0.0
        # print("Ethics module state reset.") # Suppress print for cleaner output


class SimpleMockWrittingModule:
    """A simple mock module simulating writing quality and progress."""
    def __init__(self, config: Dict[str, Any]):
        # Access config parameters based on RLConfig schema keys
        self.config = config
        self.writing_quality: float = 0.0 # State variable: Estimated quality (0.0 to 1.0)
        self.thesis_quality: float = 0.0  # State variable: Overall thesis quality (0.0 to 1.0)
        self.language_proficiency: float = 0.5 # State variable: Language skill (0.0 to 1.0) - initialized

        # Access reward config items using the correct key 'reward_config'
        # Access the Pydantic RewardItem instance and its 'value' attribute
        reward_config = self.config.get("reward_config", {})
        fluency_improved_reward_item = reward_config.get("fluency_improved")
        self._fluency_reward_value = getattr(fluency_improved_reward_item, "value", 1.0) if fluency_improved_reward_item else 1.0

        # Access action effects config using the correct key 'action_effects'
        # This access pattern might need adjustment depending on how action_effects are used outside env.step
        # For now, assume it's used by the environment to update state, not directly here.


    def update(self, action: str, log: List[str]):
        """Simulate state update based on action and log."""
        # Example: Actions 'write_0' improves writing quality
        if action == "write_0":
            self.writing_quality = np.clip(self.writing_quality + 0.1, 0.0, 1.0)
            self.thesis_quality = np.clip(self.thesis_quality + 0.05, 0.0, 1.0)
            log.append("Writing: Section rewritten.")
        # Example: Action 'write_1' improves thesis quality by reforming outline
        elif action == "write_1":
             self.thesis_quality = np.clip(self.thesis_quality + 0.07, 0.0, 1.0)
             log.append("Writing: Outline reformed.")
        # Example: Writing quality naturally fluctuates or decays slightly
        self.writing_quality = np.clip(self.writing_quality * 0.99, 0.0, 1.0)


    def reset(self):
        """Reset module state for a new episode."""
        self.writing_quality = 0.0
        self.thesis_quality = 0.0
        self.language_proficiency = 0.5 # Reset to initial value
        # print("Writing module state reset.") # Suppress print for cleaner output


class SimpleMockEmotionModule:
    """A simple mock module simulating student emotional state and autonomy."""
    def __init__(self, config: Dict[str, Any]):
        # Access config parameters based on RLConfig schema keys
        self.config = config
        self.emotional_state: float = 0.5 # State variable: Student's emotional well-being (0.0 to 1.0, higher is better)
        self.student_autonomy: float = 0.5 # State variable: Level of self-direction (0.0 to 1.0)
        self.deadline_ratio: float = 0.0   # State variable: Progress towards deadline (0.0 to 1.0) - initialized

        # Access reward config items using the correct key 'reward_config'
        # Access the Pydantic RewardItem instance and its 'value' attribute
        reward_config = self.config.get("reward_config", {})
        autonomy_respected_reward_item = reward_config.get("autonomy_respected")
        self._autonomy_reward_value = getattr(autonomy_respected_reward_item, "value", 1.0) if autonomy_respected_reward_item else 1.0

        # Access other config parameters as needed


    def update(self, action: str, log: List[str]):
        """Simulate state update based on action and log."""
        # Example: Actions 'emo_0' encourages autonomy
        if action == "emo_0":
            self.student_autonomy = np.clip(self.student_autonomy + 0.1, 0.0, 1.0)
            log.append("Emotion: Autonomy encouraged.")
        # Example: Actions 'emo_1' acknowledges stress
        elif action == "emo_1":
             self.emotional_state = np.clip(self.emotional_state - 0.05, 0.0, 1.0) # Small decrease in stress (increase in well-being)
             log.append("Emotion: Deadline stress acknowledged.")
        # Example: Timestep influences deadline ratio (simulate progress)
        # In a real env, this might be based on actual progress or a separate timer
        max_steps = self.config.get("max_episode_steps", 100)
        self.deadline_ratio = np.clip(self.config.get("timestep", 0) / max_steps, 0.0, 1.0) # Use timestep from config or env? env has timestep.
        # Using a simple linear progression for now, or could get from env state directly?
        # The environment updates the timestep, so perhaps the module should access env.timestep
        # Let's update this to potentially access env state if needed, but for now use a placeholder.
        # For this mock, let's assume deadline_ratio is updated externally or based on simplified logic.
        # Let's add a simple time-based increase for deadline_ratio for now.
        self.deadline_ratio = np.clip(self.deadline_ratio + 0.01, 0.0, 1.0) # Simulate time passing


        # Emotional state naturally fluctuates
        self.emotional_state = np.clip(self.emotional_state + (random.random() - 0.5) * 0.02, 0.0, 1.0) # Random fluctuation


    def reset(self):
        """Reset module state for a new episode."""
        self.emotional_state = 0.5
        self.student_autonomy = 0.5
        self.deadline_ratio = 0.0
        # print("Emotion module state reset.") # Suppress print for cleaner output


class SimpleMockIdeaModule:
    """A simple mock module simulating idea generation and creativity."""
    def __init__(self, config: Dict[str, Any]):
        # Access config parameters based on RLConfig schema keys
        self.config = config
        self.creativity_score: float = 0.0 # State variable: Metric for novelty/originality (0.0 to 1.0)
        self.brainstorming_score: float = 0.0 # State variable: Metric for brainstorming activity (0.0 to 1.0)
        self.advisor_feedback: float = 0.5 # State variable: Advisor feedback/trust (0.0 to 1.0) - initialized
        self.thesis_difficulty: float = 0.5 # State variable: Perceived difficulty (0.0 to 1.0) - initialized

        # Access reward config items using the correct key 'reward_config'
        # Access the Pydantic RewardItem instance and its 'value' attribute
        reward_config = self.config.get("reward_config", {})
        creativity_expressed_reward_item = reward_config.get("creativity_expressed")
        self._creativity_reward_value = getattr(creativity_expressed_reward_item, "value", 1.0) if creativity_expressed_reward_item else 1.0

        # Access other config parameters as needed


    def update(self, action: str, log: List[str]):
        """Simulate state update based on action and log."""
        # Example: Actions 'brain_0' increases creativity
        if action == "brain_0":
            self.creativity_score = np.clip(self.creativity_score + 0.05, 0.0, 1.0)
            self.brainstorming_score = np.clip(self.brainstorming_score + 0.1, 0.0, 1.0)
            log.append("Idea: Open-ended reflection prompted.")
        # Example: Actions 'brain_2' stimulates cross-topic merge
        elif action == "brain_2":
             self.creativity_score = np.clip(self.creativity_score + 0.1, 0.0, 1.0)
             log.append("Idea: Cross-topic merge stimulated.")
        # Example: Advisor feedback changes based on actions (simplified)
        if action in ["eth_2", "write_3"]: # Actions that might improve advisor feedback
            self.advisor_feedback = np.clip(self.advisor_feedback + 0.05, 0.0, 1.0)
        elif action in ["eth_3"]: # Actions that might decrease advisor feedback
             self.advisor_feedback = np.clip(self.advisor_feedback - 0.1, 0.0, 1.0)

        # Creativity naturally decays
        self.creativity_score = np.clip(self.creativity_score * 0.95, 0.0, 1.0)


    def reset(self):
        """Reset module state for a new episode."""
        self.creativity_score = 0.0
        self.brainstorming_score = 0.0
        self.advisor_feedback = 0.5 # Reset to initial value
        self.thesis_difficulty = 0.5 # Reset to initial value
        # print("Idea module state reset.") # Suppress print for cleaner output


# Example Usage (optional, can be moved to a test cell)
if __name__ == "__main__":
    print("--- Testing Mock Modules ---")
    # Create a dummy config dictionary matching the expected structure (or load a real one)
    # This dummy config should match the RLConfig structure, especially for keys used by modules
    dummy_config = {
        "state_variables": [
            "ethical_flags", "ai_usage", "writing_quality", "thesis_quality",
            "emotional_state", "student_autonomy", "creativity_score",
            "brainstorming_score", "advisor_feedback", "deadline_ratio",
            "language_proficiency", "thesis_difficulty", "timestep"
        ],
        "actions": {
            "eth_3": "Log academic concern",
            "eth_1": "Propose AI restriction",
            "write_0": "Suggest rewriting section",
            "write_1": "Recommend outline reform",
            "emo_0": "Encourage autonomy",
            "emo_1": "Acknowledge deadline stress",
            "brain_0": "Prompt open-ended reflection",
            "brain_2": "Stimulate cross-topic merge",
            "eth_2": "Recommend advisor check-in",
            "write_3": "Enable feedback loop",
        },
        "reward_config": { # Use reward_config key
            "ethical_boundary_crossed": {"value": -6.0, "condition_type": "threshold_greater", "variable": "ethical_flags", "threshold": 0.7, "justification": "test", "risk": "test"},
            "fluency_improved": {"value": 1.5, "condition_type": "linear_scale", "variable": "writing_quality", "scaling_factor": 2.0, "justification": "test", "risk": "test"},
            "autonomy_respected": {"value": 1.0, "condition_type": "threshold_greater", "variable": "student_autonomy", "threshold": 0.5, "justification": "test", "risk": "test"},
            "creativity_expressed": {"value": 2.5, "condition_type": "linear_scale", "variable": "creativity_score", "scaling_factor": 1.5, "justification": "test", "risk": "test"},
             # Add other reward items used by _compute_reward if needed for testing modules indirectly
        },
        "action_effects": {}, # Not directly used by module.update in this mock
        "action_transitions": {}, # Not used by module.update in this mock
        "max_episode_steps": 100 # Add max_episode_steps for deadline_ratio simulation
    }

    # Create instances of mock modules
    ethics_module = SimpleMockEthicsModule(dummy_config)
    writing_module = SimpleMockWrittingModule(dummy_config)
    emotion_module = SimpleMockEmotionModule(dummy_config)
    idea_module = SimpleMockIdeaModule(dummy_config)

    print("\nInitial States:")
    print(f"Ethics Flags: {ethics_module.ethical_flags:.2f}, AI Usage: {ethics_module.ai_usage:.2f}")
    print(f"Writing Quality: {writing_module.writing_quality:.2f}, Thesis Quality: {writing_module.thesis_quality:.2f}")
    print(f"Emotional State: {emotion_module.emotional_state:.2f}, Student Autonomy: {emotion_module.student_autonomy:.2f}, Deadline Ratio: {emotion_module.deadline_ratio:.2f}")
    print(f"Creativity Score: {idea_module.creativity_score:.2f}, Brainstorming Score: {idea_module.brainstorming_score:.2f}, Advisor Feedback: {idea_module.advisor_feedback:.2f}")


    # Simulate some updates
    print("\nSimulating Updates:")
    log = []
    ethics_module.update("eth_3", log)
    writing_module.update("write_0", log)
    emotion_module.update("emo_0", log)
    idea_module.update("brain_0", log)

    print("\nStates After Updates:")
    print(f"Ethics Flags: {ethics_module.ethical_flags:.2f}, AI Usage: {ethics_module.ai_usage:.2f}")
    print(f"Writing Quality: {writing_module.writing_quality:.2f}, Thesis Quality: {writing_module.thesis_quality:.2f}")
    print(f"Emotional State: {emotion_module.emotional_state:.2f}, Student Autonomy: {emotion_module.student_autonomy:.2f}, Deadline Ratio: {emotion_module.deadline_ratio:.2f}")
    print(f"Creativity Score: {idea_module.creativity_score:.2f}, Brainstorming Score: {idea_module.brainstorming_score:.2f}, Advisor Feedback: {idea_module.advisor_feedback:.2f}")
    print("\nLog:", log)

    # Test reset
    print("\nTesting Reset:")
    ethics_module.reset()
    writing_module.reset()
    emotion_module.reset()
    idea_module.reset()

    print("\nStates After Reset:")
    print(f"Ethics Flags: {ethics_module.ethical_flags:.2f}, AI Usage: {ethics_module.ai_usage:.2f}")
    print(f"Writing Quality: {writing_module.writing_quality:.2f}, Thesis Quality: {writing_module.thesis_quality:.2f}")
    print(f"Emotional State: {emotion_module.emotional_state:.2f}, Student Autonomy: {emotion_module.student_autonomy:.2f}, Deadline Ratio: {emotion_module.deadline_ratio:.2f}")
    print(f"Creativity Score: {idea_module.creativity_score:.2f}, Brainstorming Score: {idea_module.brainstorming_score:.2f}, Advisor Feedback: {idea_module.advisor_feedback:.2f}")

    print("\n--- Mock Modules Test Complete ---")

--- Testing Mock Modules ---

Initial States:
Ethics Flags: 0.00, AI Usage: 0.00
Writing Quality: 0.00, Thesis Quality: 0.00
Emotional State: 0.50, Student Autonomy: 0.50, Deadline Ratio: 0.00
Creativity Score: 0.00, Brainstorming Score: 0.00, Advisor Feedback: 0.50

Simulating Updates:

States After Updates:
Ethics Flags: 0.20, AI Usage: 0.00
Writing Quality: 0.10, Thesis Quality: 0.05
Emotional State: 0.51, Student Autonomy: 0.60, Deadline Ratio: 0.01
Creativity Score: 0.05, Brainstorming Score: 0.10, Advisor Feedback: 0.50

Log: ['Ethics: Academic concern logged.', 'Writing: Section rewritten.', 'Emotion: Autonomy encouraged.', 'Idea: Open-ended reflection prompted.']

Testing Reset:

States After Reset:
Ethics Flags: 0.00, AI Usage: 0.00
Writing Quality: 0.00, Thesis Quality: 0.00
Emotional State: 0.50, Student Autonomy: 0.50, Deadline Ratio: 0.00
Creativity Score: 0.00, Brainstorming Score: 0.00, Advisor Feedback: 0.50

--- Mock Modules Test Complete ---


## Part 1: RL Configuration Manager

This module provides a centralized configuration interface for the RL-based educational coaching system. It manages all static settings required by the agent and environment to interpret observations, execute actions, and compute rewards.

The configuration is saved and loaded from a JSON file (`rl_config.json`). If the file does not exist, a default configuration is created automatically. This ensures a reproducible setup and simplifies testing across modules.

## Configuration Structure

- `state_variables`:  
  Defines the numerical features observed by the RL agent at each time step. These features represent the student’s academic state, ethical posture, writing progression, and contextual traits.

- `actions`:  
  A dictionary of discrete, recommendation-style actions the agent can select. Each key is an action ID (e.g. `"eth_0"`), and each value is a human-readable description.

- `reward_config`:  
  Defines the scalar reward shaping used during training. Rewards and penalties reflect fluency, ethical alignment, advisor trust, autonomy, and creativity.

- `action_effects`:  
  Maps agent actions to updates in the simulated state. Each action has associated side-effects that alter one or more student-related variables (e.g., reducing `ai_usage` or increasing `advisor_trust`).

## Usage Example

# Load existing configuration or initialize default
config = RLConfigManager.load_config()

# Access specific parts
print("Available state variables:")
for var in config["state_variables"]:
    print("-", var)
**Interaction with Other Components:**
- **Developer Dashboard:** The `DeveloperDashboard` uses `RLConfigManager` to load and save the configuration, allowing developers to interactively modify the settings via a Streamlit interface.
- **Data Preprocessor:** The `DataPreprocessor` uses the configuration (specifically, `state_variables` and `reward_config`) to determine how to convert raw usage logs into state vectors and compute rewards.
- **RL Environment:** The `SupervisorEnv` is built dynamically based on the configuration loaded by `RLConfigManager`, defining its observation space, action space, and state transition logic (`action_effects`).
- **RL Training Loop and Trainer:** These components implicitly rely on the configuration loaded by `RLConfigManager` via the environment and preprocessor.

**Data Structures and Configuration:**
The configuration is stored in a JSON file (`rl_config.json`) and represented in Python as a dictionary with the following structure:
- "state_variables": A list of strings, where each string is the name of a variable that constitutes the state observed by the RL agent.
- "actions": A list of strings, where each string is a human-readable label for an action the RL agent can take. The index of the action in this list is the action ID used by the RL model.
- "reward_config": A dictionary mapping event names (as found in usage logs) to numerical reward values.
- "action_effects": A nested dictionary defining how actions change state variables. The outer keys are string representations of action indices, and the inner dictionaries map state variable names to the delta value (change) applied to that variable when the action is taken.


In [221]:
# PART 1: RL CONFIGURATION MANAGER
# ===========================================================
#
# This module is responsible for loading and managing the RL
# configuration, which defines the state space, action space,
# reward function, and state transition dynamics.
#
# - Centralized configuration management.
# - Supports loading from different sources (e.g., file, dictionary).
# - Provides validation for the configuration structure.
#
# -----------------------------------------------------------
# ✅ WHY CENTRALIZED CONFIGURATION?
# -----------------------------------------------------------
# - Single source of truth for RL parameters.
# - Easy to modify and experiment with different configurations.
# - Decouples RL logic from specific environment/task details.
# - Allows state to be distributed across multiple simulated modules.
#
# -----------------------------------------------------------
# ✅ KEY CONCEPTS:
# -----------------------------------------------------------
# - State Variables: Define the observation space.
# - Actions: Define the action space.
# - Reward Config: Define components of the reward function.
# - Action Effects: Define how actions change state variables.
# - Action Transitions: Define basic next-state routing (for non-RL graph components).
# - Module-Specific Configs: Parameters needed by individual mock modules.
# ===========================================================

from typing import Dict, List, Any, TypedDict, NamedTuple
import json
import os
import datetime
from pydantic import BaseModel, Field, ValidationError, model_validator
import yaml # Assuming PyYAML is installed for config file handling

# Define the data structures for configuration validation
class RewardItem(BaseModel):
    """Defines parameters for a single reward component."""
    value: float = Field(..., description="The base value of the reward component.")
    condition_type: str = Field(..., description="The type of condition to trigger the reward (e.g., 'threshold_greater', 'event_present').")
    variable: str = Field(..., description="The state variable or log event key this reward relates to.")
    threshold: float = Field(0.0, description="Threshold value for condition types like 'threshold_greater' or 'threshold_less'.")
    scaling_factor: float = Field(1.0, description="Factor to scale the reward value, especially for 'linear_scale' conditions.")
    justification: str = Field(..., min_length=10, description="Justification for why this reward component is included.")
    risk: str = Field(..., min_length=5, description="Associated risk level for this reward component (e.g., 'Low', 'Medium', 'High').")

class ActionEffects(BaseModel):
    """Defines how a single action affects multiple state variables."""
    # Uses a dictionary where keys are state variable names and values are the delta changes.
    # Using a named field 'effects'
    effects: Dict[str, float] = Field(..., description="Dictionary of state variable changes.")

    # @model_validator(mode='after')
    # def check_variables_exist(self):
    #     # In a real scenario, you'd validate that these variable names exist in state_variables
    #     return self


class ActionTransitions(BaseModel):
    """Defines potential next steps or states after an action."""
    # Simple example: mapping to next action keys for a linear chain
    next: str = Field(None, description="The key of the next action or node in a sequence.")
    # Could be expanded for conditional transitions, probabilities, etc.


class RLConfig(BaseModel):
    """Schema for the main Reinforcement Learning configuration."""
    config_version: float = Field(1.0, description="Version of the configuration schema.")
    created_at: datetime.datetime = Field(default_factory=datetime.datetime.now, description="Timestamp of configuration creation.")
    state_variables: List[str] = Field(..., description="List of state variable names defining the observation space.")
    actions: Dict[str, str] = Field(..., description="Dictionary mapping action keys (str) to human-readable labels (str).")
    reward_config: Dict[str, RewardItem] = Field(..., description="Dictionary defining reward components, keyed by reward name.")
    # Expecting Dict[str, ActionEffects] where ActionEffects has a named field 'effects'
    action_effects: Dict[str, ActionEffects] = Field(default_factory=dict, description="Dictionary mapping action keys to their effects on state variables.")
    action_transitions: Dict[str, ActionTransitions] = Field(default_factory=dict, description="Dictionary mapping action keys to their transition rules.")

    # Add module-specific configuration parameters here
    ethics_threshold: float = Field(0.7, description="Threshold for ethical flags to trigger warnings.")
    # Add other module-specific parameters as needed (e.g., writing_quality_decay_rate, emotion_fluctuation_range)


    # Custom validation to ensure action_effects and action_transitions keys exist in actions
    @model_validator(mode='after')
    def validate_action_keys(self):
        action_keys = set(self.actions.keys())
        effect_keys = set(self.action_effects.keys())
        transition_keys = set(self.action_transitions.keys())

        if not effect_keys.issubset(action_keys):
            missing_in_actions = effect_keys - action_keys
            raise ValueError(f"Action effects defined for keys not in actions: {missing_in_actions}")

        if not transition_keys.issubset(action_keys):
             missing_in_actions = transition_keys - action_keys
             raise ValueError(f"Action transitions defined for keys not in actions: {missing_in_actions}")

        return self

class RLConfigManager:
    """
    Manages loading, validating, and providing the RL configuration.
    """
    def __init__(self, config_source=None):
        """
        Initialize the Config Manager.

        Args:
            config_source (str or dict, optional): Source of the configuration.
                If a string, treated as a file path. If a dict, used directly.
                If None, attempts to load from a default location or uses a default config.
        """
        self.config_source = config_source
        self._config = None # Store the loaded and validated config


    def load_config(self) -> Dict[str, Any]:
        """
        Loads the configuration from the specified source or uses a default.

        Returns:
            Dict[str, Any]: The loaded and validated configuration dictionary,
                            with nested Pydantic models retained for specific fields.

        Raises:
            ValidationError: If the loaded configuration does not match the schema.
            FileNotFoundError: If the config_source file does not exist.
            json.JSONDecodeError or yaml.YAMLError: If the config file is invalid.
        """
        raw_config = None

        if isinstance(self.config_source, dict):
            print("Loading configuration from dictionary source.")
            raw_config = self.config_source
        elif isinstance(self.config_source, str):
            print(f"Loading configuration from file: {self.config_source}")
            if not os.path.exists(self.config_source):
                raise FileNotFoundError(f"Configuration file not found: {self.config_source}")
            with open(self.config_source, 'r') as f:
                # Assume YAML for flexibility, can add JSON handling
                raw_config = yaml.safe_load(f)
        else:
            print("No config source provided, using default configuration.")
            raw_config = self._get_default_config()

        # Validate the loaded configuration against the schema
        try:
            # Use Pydantic to parse and validate the raw config dictionary
            validated_config = RLConfig(**raw_config)

            # Manually construct the output dictionary to retain nested Pydantic models
            # for action_effects, reward_config, and action_transitions.
            # Other fields can be dumped to their standard Python types.
            output_config = validated_config.model_dump(exclude={
                'action_effects',
                'reward_config',
                'action_transitions'
            })

            # Add the Pydantic instances back for the specified keys
            # Access the validated Pydantic models directly from validated_config object
            output_config['action_effects'] = validated_config.action_effects
            output_config['reward_config'] = validated_config.reward_config
            output_config['action_transitions'] = validated_config.action_transitions

            # Include other top-level fields directly from the validated config object
            output_config['config_version'] = validated_config.config_version
            output_config['created_at'] = validated_config.created_at
            output_config['state_variables'] = validated_config.state_variables
            output_config['actions'] = validated_config.actions
            output_config['ethics_threshold'] = validated_config.ethics_threshold # Include the new field
            # Include other top-level fields here as they are added to RLConfig


            self._config = output_config # Store the custom-built config
            print("✅ Configuration validated and loaded successfully (retaining Pydantic instances).")
            return self._config

        except ValidationError as e:
            print("❌ Configuration validation failed.")
            raise e
        except Exception as e:
             print(f"❌ An unexpected error occurred during config loading/validation: {e}")
             raise e


    def get_config(self) -> Dict[str, Any]:
        """
        Returns the currently loaded and validated configuration.

        If config hasn't been loaded, it will attempt to load it.
        """
        if self._config is None:
            self.load_config()
        return self._config

    def _get_default_config(self) -> Dict[str, Any]:
        """
        Provides a default RL configuration dictionary.

        This is used if no config_source is provided.
        """
        print("Using default configuration.")
        # Define a default configuration dictionary that matches the RLConfig schema
        # Ensure RewardItem values are dictionaries matching the RewardItem schema
        # Ensure ActionEffects values are dictionaries matching the ActionEffects schema { "effects": { ... } }
        # Ensure ActionTransitions values are dictionaries that Pydantic will convert to ActionTransitions instances { "next": ... }

        # Define action effects data in the format expected by ActionEffects BaseModel { "effects": { ... } }
        default_action_effects_data = {
            "eth_0": {"effects": {"ethical_flags": -0.1}}, # Decrease ethical flags
            "eth_1": {"effects": {"ai_usage": -0.2}}, # Decrease AI usage
            "eth_2": {"effects": {"advisor_feedback": 0.15}}, # Increase advisor feedback/trust
            "eth_3": {"effects": {"ethical_flags": 0.2, "advisor_feedback": -0.3}}, # Increase ethical flags, decrease advisor feedback
            "brain_0": {"effects": {"creativity_score": 0.05}}, # Increase creativity
            "brain_1": {"effects": {"creativity_score": 0.07}}, # Increase creativity more
            "brain_2": {"effects": {"creativity_score": 0.1}}, # Increase creativity even more
            "brain_3": {"effects": {"thesis_quality": 0.05}}, # Small increase in thesis quality
            "write_0": {"effects": {"thesis_quality": 0.1}}, # Increase thesis quality
            "write_1": {"effects": {"thesis_quality": 0.07}}, # Increase thesis quality less
            "write_2": {"effects": {"thesis_quality": 0.05}}, # Small increase in thesis quality
            "write_3": {"effects": {"thesis_quality": 0.05, "advisor_feedback": 0.1}}, # Increase quality and advisor feedback
            "emo_0": {"effects": {"student_autonomy": 0.1}}, # Increase student autonomy
            "emo_1": {"effects": {"emotional_state": -0.05}}, # Decrease emotional state (reduce stress)
            "emo_2": {"effects": {"emotional_state": -0.1}}, # Decrease emotional state more
            "emo_3": {"effects": {"emotional_state": 0.05, "student_autonomy": -0.05}}, # Increase emotional state (positive boost) but slight autonomy decrease (if guided)
            # Add effects for other actions here
        }

        # Define action transitions data as dictionaries matching the ActionTransitions schema { "next": ... }
        default_action_transitions_data = {
            "eth_0": {"next": "write_0"},
            "write_0": {"next": "emo_0"},
            "emo_0": {"next": "brain_0"},
            "brain_0": {"next": "END"} # Example sequence end
            # Add transitions for other actions here
        }

        default_config = {
            "config_version": 1.1,
            "created_at": datetime.datetime.now().isoformat(), # Use ISO format for JSON compatibility
            "state_variables": [
                "embedding_drift", "ai_usage", "ethical_flags", "advisor_feedback", "timestep",
                "brainstorming_score", "writing_quality", "emotion_state", "temporary_test_variable",
                # Add other variables here as needed for your modules
                "creativity_score", "student_autonomy", "thesis_quality", "emotional_state", # Added from mock modules
                "language_proficiency", "thesis_difficulty", # Example additional state vars
                "stage" # Example categorical state variable (will need encoding for RL)
            ],
            "actions": {
                "eth_0": "Display ethical reminder",
                "eth_1": "Propose AI restriction",
                "eth_2": "Recommend advisor check-in",
                "eth_3": "Log academic concern",
                "brain_0": "Prompt open-ended reflection",
                "brain_1": "Offer question inversion",
                "brain_2": "Stimulate cross-topic merge",
                "brain_3": "Show novelty heatmap",
                "write_0": "Suggest rewriting section",
                "write_1": "Recommend outline reform",
                "write_2": "Display writing tip",
                "write_3": "Enable feedback loop",
                "emo_0": "Encourage autonomy",
                "emo_1": "Acknowledge deadline stress",
                "emo_2": "Suggest micro break",
                "emo_3": "Offer motivational boost",
                # Add other actions as needed
            },
            "reward_config": {
                # Define each reward component as a dictionary matching the RewardItem schema
                "fluency_improved": {
                    "value": 1.5,
                    "condition_type": "linear_scale",
                    "variable": "writing_quality", # Example: Reward scales with writing quality
                    "scaling_factor": 2.0, # Scale factor
                    "justification": "Encourage high quality and fluent writing.",
                    "risk": "Low Risk"
                },
                "trust_earned": {
                    "value": 2.0,
                    "condition_type": "threshold_greater",
                    "variable": "advisor_feedback", # Example: Reward if advisor feedback is high
                    "threshold": 0.7,
                    "justification": "Incentivize actions that improve advisor trust.",
                    "risk": "Medium Risk"
                },
                 "creativity_expressed": {
                     "value": 2.5,
                     "condition_type": "linear_scale",
                     "variable": "creativity_score", # Example: Reward scales with creativity
                     "scaling_factor": 1.5,
                     "justification": "Promote creative and novel ideas.",
                     "risk": "Low Risk"
                 },
                 "autonomy_respected": {
                     "value": 1.0,
                     "condition_type": "threshold_greater",
                     "variable": "student_autonomy", # Example: Reward if student autonomy is high
                     "threshold": 0.5,
                     "justification": "Encourage actions that respect student autonomy.",
                     "risk": "Low Risk"
                 },
                 "ai_dependency_violation": {
                     "value": -4.0,
                     "condition_type": "threshold_greater",
                     "variable": "ai_usage", # Example: Penalty for high AI usage
                     "threshold": 0.8,
                     "justification": "Penalize excessive reliance on AI.",
                     "risk": "High Risk"
                 },
                 "ethical_boundary_crossed": {
                     "value": -6.0,
                     "condition_type": "threshold_greater",
                     "variable": "ethical_flags", # Example: Penalty for high ethical flags
                     "threshold": 0.7,
                     "justification": "Strongly penalize actions leading to ethical issues.",
                     "risk": "Very High Risk"
                 },
                 "deadline_panic_detected": {
                     "value": -1.0,
                     "condition_type": "event_present",
                     "variable": "deadline_panic_detected", # Example: Penalty if panic event detected in log
                     "justification": "Penalize situations indicating poor time management.",
                     "risk": "Medium Risk"
                 },
                 "milestone_completed": {
                     "value": 5.0,
                     "condition_type": "event_present",
                     "variable": "milestone_completed", # Example: Bonus if milestone completed event in log
                     "justification": "Reward achieving key project milestones.",
                     "risk": "Low Risk"
                 },
                 "novel_but_safe": {
                     "value": 3.0,
                     "condition_type": "threshold_greater",
                     "variable": "creativity_score",
                     "threshold": 0.6,
                     # Example of combining conditions conceptually (needs logic in _compute_reward)
                     # Here simplified to just check creativity score
                     "justification": "Bonus for highly creative ideas that do not raise ethical flags.",
                     "risk": "Medium Risk"
                 },
                  "supervisor_disappointment": {
                      "value": -5.0,
                      "condition_type": "threshold_less",
                      "variable": "advisor_feedback", # Example: Penalty for low advisor feedback
                      "threshold": 0.2,
                      "justification": "Penalize actions that lead to advisor dissatisfaction.",
                      "risk": "High Risk"
                  }
                # Add other reward items here
            },
             "action_effects": default_action_effects_data, # Use the pre-defined dictionary
             "action_transitions": default_action_transitions_data, # Use the pre-defined dictionary data (Pydantic converts)
             "ethics_threshold": 0.7 # Add the new ethics threshold parameter
        }
        return default_config

# Example Usage (optional, can be moved to a test cell)
if __name__ == "__main__":
    print("--- Testing RLConfigManager ---")
    # Test loading default config
    try:
        mgr_default = RLConfigManager()
        config_default = mgr_default.load_config()
        print("\nLoaded Default Config (sample):")
        print("Config Version:", config_default.get("config_version"))
        print("Ethics Threshold:", config_default.get("ethics_threshold")) # Access the new field
        print("State Variables:", config_default.get("state_variables", [])[:5])
        print("Actions (keys):", list(config_default.get("actions", {}).keys())[:5])
        print("Reward Config (keys):", list(config_default.get("reward_config", {}).keys())[:5])
        # Verify a RewardItem instance type
        sample_reward_item = list(config_default.get("reward_config", {}).values())[0]
        print(f"Sample Reward Item Type: {type(sample_reward_item)}")
        print(f"Sample Reward Item Value: {getattr(sample_reward_item, 'value', 'N/A')}")

        # Verify an ActionEffects instance type and its content
        sample_action_effect_key = list(config_default.get("action_effects", {}).keys())[0]
        sample_action_effect = config_default.get("action_effects", {}).get(sample_action_effect_key)
        print(f"Sample Action Effect Type ('{sample_action_effect_key}'): {type(sample_action_effect)}")
        # When using a named field, access the field
        print(f"Sample Action Effect Effects Dict ('{sample_action_effect_key}'): {getattr(sample_action_effect, 'effects', 'N/A')}")


        # Verify an ActionTransitions instance type and its content
        sample_action_transition_key = list(config_default.get("action_transitions", {}).keys())[0]
        sample_action_transition = config_default.get("action_transitions", {}).get(sample_action_transition_key)
        print(f"Sample Action Transition Type ('{sample_action_transition_key}'): {type(sample_action_transition)}")
        print(f"Sample Action Transition Next ('{sample_action_transition_key}'): {getattr(sample_action_transition, 'next', 'N/A')}")


    except Exception as e:
        print(f"\n❌ Error during Config Manager test: {e}")

    print("\n--- Config Manager Test Complete ---")

--- Testing RLConfigManager ---
No config source provided, using default configuration.
Using default configuration.
✅ Configuration validated and loaded successfully (retaining Pydantic instances).

Loaded Default Config (sample):
Config Version: 1.1
Ethics Threshold: 0.7
State Variables: ['embedding_drift', 'ai_usage', 'ethical_flags', 'advisor_feedback', 'timestep']
Actions (keys): ['eth_0', 'eth_1', 'eth_2', 'eth_3', 'brain_0']
Reward Config (keys): ['fluency_improved', 'trust_earned', 'creativity_expressed', 'autonomy_respected', 'ai_dependency_violation']
Sample Reward Item Type: <class '__main__.RewardItem'>
Sample Reward Item Value: 1.5
Sample Action Effect Type ('eth_0'): <class '__main__.ActionEffects'>
Sample Action Effect Effects Dict ('eth_0'): {'ethical_flags': -0.1}
Sample Action Transition Type ('eth_0'): <class '__main__.ActionTransitions'>
Sample Action Transition Next ('eth_0'): write_0

--- Config Manager Test Complete ---


## Part 2: RL Environment

The `SupervisorEnv` class is a custom Reinforcement Learning environment built using the Gymnasium library. It is designed to simulate the interaction between the Thesis Assistant's ethics module and a student's thesis writing process, allowing an RL agent to learn optimal intervention policies. A key feature is its dynamic nature, where the state space, action space, rewards, and state transitions are defined by the loaded configuration.

**Purpose:**
To provide a simulated environment where the RL agent (the Ethics Supervisor) can learn through trial and error. The environment presents states representing the current situation (ethical flags, AI usage, progress, etc.) and provides feedback (rewards) based on the agent's chosen actions and the resulting changes in the simulated student's state.

**Key Components:**
- `__init__(ethics_module, config)`: Initializes the environment. It takes a simulated or actual `ethics_module` object (which holds the state variables) and the RL configuration. It dynamically defines the `observation_space` and `action_space` based on the configuration.
- `reset(seed=None)`: Resets the environment to an initial state at the beginning of a new simulation episode. It also resets the internal timestep counter.
- `step(action)`: Executes one step in the environment based on the `action` taken by the RL agent. It computes a reward, applies the effects of the action to the state (using `_apply_action_effects`), increments the timestep, and determines if the episode is done.
- `_get_state()`: Constructs the current state vector observed by the agent. It gathers the values of the state variables from the `ethics_module` object based on the configuration and normalizes them.
- `_compute_reward(action)`: Calculates the reward signal for the current step. The provided implementation is a placeholder that considers the timestep and a simple cost related to the action index. In a more complete system, this would incorporate ethical outcomes, user feedback, advisor input, etc.
- `_apply_action_effects(action)`: Updates the state variables in the `ethics_module` object based on the effects defined for the taken action in the configuration.

**How to Use:**
- Initialize the environment: `env = SupervisorEnv(mock_ethics_module_instance, config)`. The `mock_ethics_module_instance` should be an object (like `MockEthicsModule`) that holds the current values of the state variables defined in the config.
- Call `env.reset()` to start a new episode.
- In a training or inference loop, get an action from the RL agent and call `env.step(action)` to advance the simulation. The `step` method returns the next state, reward, and episode status.

**Interaction with Other Components:**
- **Mock Ethics Module:** The environment directly reads state variable values from and writes updated values to an instance of `MockEthicsModule` (or a similar object representing the system state).
- **Configuration Manager:** The environment's fundamental structure (state and action spaces, action effects) is determined by the configuration loaded by `RLConfigManager`.
- **PPO Supervisor (RL Agent):** The `SupervisorRL` class interacts with the `SupervisorEnv` to train the PPO model by calling `reset()` and `step()` and receiving state and reward information.
- **Data Preprocessor:** While not directly used by the environment during a `step`, the state variables and reward structure defined in the configuration used by the environment are consistent with what the `DataPreprocessor` expects when processing raw logs.

**Data Structures and Configuration:**
- The environment's state is represented as a NumPy array (`observation_space`).
- Actions are discrete integers (`action_space`).
- It uses the `state_variables`, `actions`, and `action_effects` dictionaries from the RL configuration to define its behavior.
- The `_compute_reward` method is a placeholder and would ideally use the `reward_config` from the configuration and events from the `ethics_module` state.

In [222]:
# PART 2: RL ENVIRONMENT (Fully Dynamic Gym-Compatible Environment)
# ===========================================================
#
# This module defines the RL environment the PPO agent interacts with.
#
# - Fully config-driven:
#   - The state space (variables used)
#   - The action space (interventions)
#   - The reward function
#   - The state update effects per action
#
# -----------------------------------------------------------
# ✅ WHY FULLY DYNAMIC STATE?
# -----------------------------------------------------------
# - Allows easy expansion of system complexity.
# - Developers can add new state features via config without touching any code.
# - Keeps RL model compatible with evolving assistant behavior.
# - Allows state to be distributed across multiple simulated modules.
#
# -----------------------------------------------------------
# ✅ KEY CONCEPTS (NOW FULLY DYNAMIC):
# -----------------------------------------------------------
# - State Variables: loaded from `state_variables` in config
# - Action Effects: loaded from `action_effects` in config
# - Observation Space: dynamically computed based on config
# - Multi-module state management: Environment interacts with multiple module instances.
# - Dynamic Reward: Reward computed based on config and module states/log events.
# ===========================================================

# Assuming TypedDict, List, Dict, Any, np, random, gym are imported in previous cells.
# Assuming mock module classes (SimpleMockEthicsModule, etc.) are defined in previous cells.
# Assuming ThesisState TypedDict is defined in a previous cell.
# Assuming RewardItem and ActionEffects BaseModel are defined in the config manager cell.

from typing import NamedTuple # Keep NamedTuple for old reference if needed, but using BaseModel
from pydantic import BaseModel # Ensure BaseModel is imported

# Redefine RewardItem and ActionEffects just in case, should be synced with config manager
class RewardItem(BaseModel):
    value: float
    condition_type: str
    variable: str
    threshold: float = 0.0
    scaling_factor: float = 1.0
    justification: str = ""
    risk: str = ""

class ActionEffects(BaseModel):
    effects: Dict[str, float] # Inner dictionary of effects


class CoreState(TypedDict):
    """
    Core state variables for the thesis assistant.

    These variables represent the student’s current status across different
    dimensions relevant to the RL policy.
    """
    stage: str # Current stage of the thesis (e.g., planning, drafting, editing)
    advisor_trust: float # Level of trust from the advisor (e.g., 0.0 to 1.0)
    creativity_score: float # Metric for the novelty and originality of ideas (e.g., 0.0 to 1.0)
    ethical_flags: float # Aggregated score indicating potential ethical concerns (e.g., 0.0 to 1.0, higher is worse)
    ai_usage: float # Metric for the level of AI assistance used (e.g., 0.0 to 1.0)
    thesis_quality: float # Estimated quality of the thesis content (e.g., 0.0 to 1.0)
    deadline_ratio: float # Progress towards the deadline (e.g., 0.0 to 1.0)
    thesis_difficulty: float # Perceived difficulty of the thesis topic/task (e.g., 0.0 to 1.0)
    student_autonomy: float # Level of student self-direction and initiative (e.g., 0.0 to 1.0)
    language_proficiency: float # Assessment of writing and language skills (e.0 to 1.0)
    emotional_state: float # Proxy for the student's emotional well-being (e.g., 0.0 to 1.0, higher is better)
    timestep: int # Current simulation timestep

class ThesisState(TypedDict):
    """
    Full state definition for the LangGraph.

    Combines core state variables with policy execution trace, logs,
    and the configuration dictionary.
    """
    core: Dict[str, float] # The core state variables
    policy_trace: List[str] # List of action keys executed in sequence
    log: List[str] # Log of messages or events generated by actions
    config: Dict[str, Any] # The loaded RL configuration
    env: Any # Hold the SupervisorEnv instance (using Any as it's defined later)
    done: bool # Flag indicating if the episode is finished
    truncated: bool # Flag indicating if the episode was truncated

class SupervisorEnv(gym.Env):
    """
    Fully dynamic Gym-compatible RL environment for the Thesis Assistant Supervisor.

    This environment simulates the state of a student's thesis progress and ethical
    interactions, allowing an RL agent to learn intervention policies. The environment's
    structure (state space, action space, rewards, and transitions) is dynamically
    defined by the provided configuration dictionary.

    Attributes:
        module_instances (Dict[str, Any]): A dictionary holding instances of all
                                           mock modules (e.g., ethics, writing, emotion, idea).
        config (dict): The loaded RL configuration dictionary.
        state_variables (list): List of state variable names defined in the config.
        actions (list): List of available action labels defined in the config.
        action_effects (dict): Dictionary mapping action keys to their effects on state variables (from config).
                               Values are ActionEffects instances if loaded via Pydantic, or dicts otherwise.
        reward_config (Dict[str, RewardItem]): Dictionary defining reward components.
                               Values are RewardItem instances if loaded via Pydantic, or dicts otherwise.
        observation_space (gym.spaces.Box): Dynamically computed continuous state space.
        action_space (gym.spaces.Discrete): Discrete action space based on the number of actions.
        timestep (int): The current simulation step count within an episode.
    """

    def __init__(self, module_instances: Dict[str, Any], config: Dict[str, Any]):
        """
        Initialize the environment.

        Args:
            module_instances (Dict[str, Any]): A dictionary containing instances of
                                               all mock modules (e.g., {"ethics": ethics_module_instance}).
            config (dict): The full RL configuration dictionary.
        """
        super().__init__()

        self.module_instances = module_instances
        self.config = config

        print("\n--- SupervisorEnv Initialization Debug ---")
        print(f"Received config keys: {list(self.config.keys())}")
        print(f"Config type: {type(self.config)}")

        # Access and print type/sample content of critical config sections
        action_effects_cfg = self.config.get("action_effects", {})
        reward_config_cfg = self.config.get("reward_config", {})
        actions_cfg = self.config.get("actions", {})
        state_variables_cfg = self.config.get("state_variables", [])

        print(f"Type of config['action_effects']: {type(action_effects_cfg)}")
        if isinstance(action_effects_cfg, dict):
             print(f"Sample action_effects keys: {list(action_effects_cfg.keys())[:5]}")
             if action_effects_cfg:
                  sample_key = list(action_effects_cfg.keys())[0]
                  sample_value = action_effects_cfg[sample_key]
                  print(f"Type of sample action_effects value ('{sample_key}'): {type(sample_value)}")
                  if hasattr(sample_value, 'model_dump_json'): # Check if Pydantic model
                       print(f"Sample action_effects value (model_dump_json): {sample_value.model_dump_json()}")
                  else:
                       print(f"Sample action_effects value: {sample_value}")

        print(f"Type of config['reward_config']: {type(reward_config_cfg)}")
        if isinstance(reward_config_cfg, dict):
             print(f"Sample reward_config keys: {list(reward_config_cfg.keys())[:5]}")
             if reward_config_cfg:
                  sample_key = list(reward_config_cfg.keys())[0]
                  sample_value = reward_config_cfg[sample_key]
                  print(f"Type of sample reward_config value ('{sample_key}'): {type(sample_value)}")
                  if hasattr(sample_value, 'model_dump_json'): # Check if Pydantic model
                       print(f"Sample reward_config value (model_dump_json): {sample_value.model_dump_json()}")
                  else:
                       print(f"Sample reward_config value: {sample_value}")

        print(f"Type of config['actions']: {type(actions_cfg)}")
        if isinstance(actions_cfg, dict):
             print(f"Number of actions: {len(actions_cfg)}")

        print(f"Type of config['state_variables']: {type(state_variables_cfg)}")
        if isinstance(state_variables_cfg, list):
             print(f"Number of state_variables: {len(state_variables_cfg)}")
             print(f"Sample state_variables: {state_variables_cfg[:5]}")

        print("--- End SupervisorEnv Initialization Debug ---")


        self.state_variables = state_variables_cfg
        self.actions = actions_cfg
        self.action_effects = action_effects_cfg # Assign the potentially Pydantic-containing dict
        self.reward_config = reward_config_cfg # Assign the potentially Pydantic-containing dict

        # Fully dynamic observation space size based on the number of state variables
        self.observation_space = gym.spaces.Box(
            low=0, high=1, shape=(len(self.state_variables),), dtype=np.float32
        )

        # Discrete action space based on the number of defined actions
        self.action_space = gym.spaces.Discrete(len(self.actions))
        self.timestep = 0

    def reset(self, seed=None):
        """
        Reset the environment at the start of a new episode.

        Resets the internal timestep and calls the reset method on all
        held mock module instances that have one.

        Args:
            seed (int, optional): Seed for random number generation. Defaults to None.

        Returns:
            tuple: A tuple containing the initial state and an info dictionary.
                   (state (np.array), info (dict))
        """
        super().reset(seed=seed) # Call the parent reset method
        self.timestep = 0
        # print("Environment timestep reset to 0.") # Suppress print for cleaner output

        # Reset all module states for a new episode
        for module_name, module_instance in self.module_instances.items():
             if hasattr(module_instance, 'reset'):
                  module_instance.reset()
                  # print(f"Module '{module_name}' state reset.") # Suppress print for cleaner output
             # else:
                 # print(f"Module '{module_name}' has no reset method.") # Suppress print for cleaner output


        return self._get_state(), {} # Return the initial state and an empty info dict

    def step(self, action):
        """
        Execute one interaction step in the environment.

        The agent selects an action, which may affect the environment's state.
        A reward is computed, and the environment transitions to the next state.

        Args:
            action (int): The action index selected by the RL agent.

        Returns:
            tuple: A tuple containing the next state, the reward, a 'done' flag,
                   a 'truncated' flag, and an info dictionary.
                   (state (np.array), reward (float), done (bool), truncated (bool), info (dict))
        """
        # Simulate generating a log entry for this step.
        # In a real system, the action execution and module interactions would
        # generate this log entry containing event flags and potentially state snapshots.
        # For this mock env, we simulate it here for reward calculation.
        current_state_values = self._get_state_values_dict() # Get current state as dictionary before effects
        simulated_log_entry = self._simulate_log_entry(current_state_values)


        # Compute the reward for the chosen action based on the current state and simulated log
        reward = self._compute_reward(action, simulated_log_entry)

        # Apply the effects of the action to the environment's state
        self._apply_action_effects(action)

        self.timestep += 1
        # Determine if the episode is finished (e.g., based on timestep limit)
        # Use max_episode_steps from config if available
        max_steps = self.config.get("max_episode_steps", 100)
        done = (self.timestep >= max_steps) # Termination condition based on max steps
        truncated = False # Example: No truncation based on external factors

        # In a real env, info might contain debug info, metrics, etc.
        info = {"simulated_log_entry": simulated_log_entry} # Include the log entry in info for potential debugging

        return self._get_state(), reward, done, truncated, info


    def _get_state(self):
        """
        Construct the normalized state vector fully dynamically.

        This method gathers the current values of the state variables from the
        various module instances based on the configuration and formats them
        into a normalized NumPy array.

        Returns:
            np.array: The normalized state vector based on config-defined variables.
        """
        state = []
        # Iterate through the state variables defined in the configuration
        for var in self.state_variables:
            if var == "timestep":
                # Normalize timestep (assuming maximum 100 steps for normalization)
                # Use max_episode_steps from config if available, otherwise default
                max_steps = self.config.get("max_episode_steps", 100)
                state.append(self.timestep / max_steps)
            else:
                # Find which module instance has this state variable and get its value
                value = 0.0 # Default value if not found
                found = False
                for module_name, module_instance in self.module_instances.items():
                    if hasattr(module_instance, var):
                        value = getattr(module_instance, var)
                        found = True
                        break # Found the variable, move to the next state variable
                if not found:
                    # print(f"Warning: State variable '{var}' not found in any initialized module instances.") # Suppress frequent warnings
                    pass # Silently skip if not found, default value 0.0 is used
                state.append(float(value)) # Ensure value is float

        return np.array(state, dtype=np.float32) # Ensure float32 dtype for compatibility with RL libraries

    def _get_state_values_dict(self):
        """
        Construct a dictionary of current raw state variable values.

        This is useful for accessing state values by name for reward calculation
        or log simulation.

        Returns:
            Dict[str, float]: Dictionary of raw state variable values.
        """
        state_values = {}
        for var in self.state_variables:
             if var == "timestep":
                  state_values[var] = self.timestep
             else:
                  value = 0.0
                  found = False
                  for module_instance in self.module_instances.values():
                       if hasattr(module_instance, var):
                            value = getattr(module_instance, var)
                            found = True
                            break
                  if not found:
                      # print(f"Warning: State variable '{var}' not found in any initialized module instances for _get_state_values_dict. Using 0.0.") # Suppress warnings
                      pass
                  state_values[var] = float(value)
        return state_values


    def _simulate_log_entry(self, current_state_values):
         """
         Simulates generating a log entry based on current state values and environment logic.

         In a real system, this would come from actual system events and module outputs.
         Here, we generate dummy event flags based on probabilities potentially
         related to state values or randomness. The keys should match variable names
         expected by the reward_config for 'event_present' conditions.

         Args:
             current_state_values (Dict[str, float]): The raw state variable values before the step.

         Returns:
             Dict[str, Any]: A dictionary simulating a log entry.
         """
         # This mock implementation needs to generate keys that are expected by the reward_config
         log_entry = {}

         # Include all current state values in the log entry for easy access by reward function
         log_entry.update(current_state_values)

         # Simulate event flags (matching reward_config keys with 'event_present' condition)
         log_entry["fluency_improved"] = random.random() < 0.1
         log_entry["trust_earned"] = random.random() < current_state_values.get("advisor_feedback", 0.0) # Higher advisor feedback increases chance
         log_entry["creativity_expressed"] = random.random() < current_state_values.get("creativity_score", 0.0) # Higher creativity increases chance
         log_entry["autonomy_respected"] = random.random() < (1.0 - current_state_values.get("ai_usage", 0.0)) # Lower AI usage increases chance
         log_entry["ai_dependency_violation"] = random.random() < current_state_values.get("ai_usage", 0.0) * 0.3 # Higher AI usage increases chance
         log_entry["ethical_boundary_crossed"] = random.random() < current_state_values.get("ethical_flags", 0.0) * 0.5 # Higher ethical flags increase chance
         log_entry["deadline_panic_detected"] = random.random() < current_state_values.get("deadline_ratio", 0.0) * 0.2 # Higher deadline ratio increases chance
         log_entry["milestone_completed"] = random.random() < 0.05 # Low probability event
         # "novel_but_safe" is defined in reward config but as threshold_greater, not event_present.
         # "supervisor_disappointment" is defined as threshold_less.

         # Add other potential log fields here that might be used by the reward function
         log_entry["prompt"] = "simulated prompt"
         log_entry["intent"] = "simulated intent"
         log_entry["thesis_stage"] = "simulated stage"


         return log_entry


    def _compute_reward(self, action, log_entry):
        """
        Compute the shaped reward for a given action based on the current state and log entry.

        This method iterates through the `reward_config` and calculates the total
        reward by summing up contributions from each defined reward item.
        It handles both Pydantic RewardItem instances and raw dictionaries.

        Args:
            action (int): The selected action index.
            log_entry (Dict[str, Any]): A dictionary simulating the log entry for this step.

        Returns:
            float: The computed reward value.
        """
        total_reward = 0.0
        reward_details = {} # Dictionary to store contributions for logging/debugging

        # Iterate through reward items defined in the config
        for reward_name, reward_item_data in self.reward_config.items():
            contribution = 0.0
            # Handle both Pydantic instances and raw dictionaries
            if isinstance(reward_item_data, RewardItem):
                 reward_item = reward_item_data
                 variable = reward_item.variable
                 condition_type = reward_item.condition_type
                 value = reward_item.value
                 threshold = reward_item.threshold
                 scaling_factor = reward_item.scaling_factor
            elif isinstance(reward_item_data, dict):
                 # Attempt to treat as a raw dictionary matching RewardItem schema
                 # This is a fallback for when Pydantic validation was bypassed
                 variable = reward_item_data.get("variable")
                 condition_type = reward_item_data.get("condition_type")
                 value = reward_item_data.get("value", 0.0)
                 threshold = reward_item_data.get("threshold", 0.0)
                 scaling_factor = reward_item_data.get("scaling_factor", 1.0)
                 # Check if essential keys are present for a dict
                 if variable is None or condition_type is None:
                      # print(f"    Warning: Reward item '{reward_name}' is a dictionary but missing essential keys (variable, condition_type). Skipping.") # Suppress warnings
                      continue # Skip this item if essential keys are missing
            else:
                 # print(f"    Warning: Reward item '{reward_name}' is of unexpected type {type(reward_item_data)}. Skipping.") # Suppress warnings
                 continue # Skip if not a Pydantic instance or dictionary


            # print(f"  Evaluating reward item '{reward_name}': variable='{variable}', condition='{condition_type}', value={value}, threshold={threshold}, scaling={scaling_factor}") # Debug print


            # Get the source value from the log entry (which includes state values and events)
            source_value = log_entry.get(variable, None) # Use None to differentiate missing from 0.0

            if source_value is None:
                # Variable not found in log entry, check if it's a state variable that wasn't included in log_entry (shouldn't happen with current _simulate_log_entry)
                # Or it's a variable defined in reward config but not in state_variables or simulated events.
                # print(f"    Warning: Reward variable '{variable}' not found in simulated log entry. Contribution is 0.") # Suppress warnings
                continue # Skip this reward item if the variable is not available

            # Calculate contribution based on condition type and source value
            if condition_type == "event_present":
                 # Expected source_value is boolean or can be interpreted as boolean
                 if bool(source_value) is True:
                      contribution = value * scaling_factor
                      # print(f"    Condition 'event_present' met (value={source_value}). Contribution: {contribution:.4f}") # Debug print
            elif condition_type == "threshold_greater":
                 # Expected source_value is numerical
                 try:
                     if float(source_value) > threshold:
                          contribution = value * scaling_factor
                          # print(f"    Condition '{variable} > {threshold}' met ({float(source_value):.4f} > {threshold}). Contribution: {contribution:.4f}") # Debug print
                 except (ValueError, TypeError):
                      # print(f"    Warning: Variable '{variable}' value ({source_value}) is not numerical for threshold_greater check. Skipping.") # Suppress warnings
                      pass # Skip if source value is not numerical
            elif condition_type == "threshold_less":
                 # Expected source_value is numerical
                 try:
                     if float(source_value) < threshold:
                          contribution = value * scaling_factor
                          # print(f"    Condition '{variable} < {threshold}' met ({float(source_value):.4f} < {threshold}). Contribution: {contribution:.4f}") # Debug print
                 except (ValueError, TypeError):
                      # print(f"    Warning: Variable '{variable}' value ({source_value}) is not numerical for threshold_less check. Skipping.") # Suppress warnings
                      pass # Skip if source value is not numerical
            elif condition_type == "linear_scale":
                 # Expected source_value is numerical
                 try:
                     # Reward scales linearly with the variable value
                     contribution = value * float(source_value) * scaling_factor
                     # print(f"    Condition 'linear_scale' applied ({value} * {float(source_value):.4f} * {scaling_factor}). Contribution: {contribution:.4f}") # Debug print
                 except (ValueError, TypeError):
                      # print(f"    Warning: Variable '{variable}' value ({source_value}) is not numerical for linear_scale. Skipping.") # Suppress warnings
                      pass # Skip if source value is not numerical
            # Add other condition types as needed (e.g., "exact_match", "range")


            total_reward += contribution
            reward_details[reward_name] = contribution # Store contribution

        # Add other potential general rewards/penalties not tied to specific reward_config items
        # Example: Simple API cost based on action index (assuming higher indices are more "costly" actions)
        api_cost = 0.002 * action
        lambda_cost = self.config.get("api_cost_lambda", 10) # Get lambda from config or use default
        api_penalty = -lambda_cost * api_cost
        total_reward += api_penalty
        reward_details["api_cost_penalty"] = api_penalty
        # print(f"  API Cost Penalty (Action {action}): {api_penalty:.4f}") # Debug print


        # Example: Time penalty to encourage finishing faster (higher penalty later)
        max_steps = self.config.get("max_episode_steps", 100)
        time_penalty = -self.config.get("time_penalty_per_step", 0.005) * (self.timestep + 1) # Get penalty from config or use default
        total_reward += time_penalty
        reward_details["time_penalty"] = time_penalty
        # print(f"  Time Penalty (Timestep {self.timestep}): {time_penalty:.4f}") # Debug print


        # print(f"--- Reward Calculation Complete (Timestep {self.timestep}, Action {action}) ---") # Suppress print
        # print(f"  Reward Details: {reward_details}") # Print all contributions
        # print(f"  Total Reward: {total_reward:.4f}") # Suppress print
        # print("--------------------------------------------------------------") # Suppress print


        return total_reward


    def _apply_action_effects(self, action):
        """
        Dynamically apply the effects of the selected action to the system state.

        Based on the 'action_effects' defined in the configuration, this method
        updates the corresponding state variables across the various module instances.
        It handles both Pydantic ActionEffects instances and raw dictionaries.

        Args:
            action (int): The selected action index.
        """
        # Get the effects defined for the chosen action (if any)
        # Ensure action is within bounds
        if action < 0 or action >= len(self.actions):
            # print(f"Warning: Invalid action index {action}. No effects applied.") # Suppress print
            return

        action_key = list(self.actions.keys())[action] # Get the action key from the index
        # Get the ActionEffects instance or raw dictionary for this action key
        action_effects_data = self.action_effects.get(action_key)

        effects = {} # Initialize effects dictionary

        # Handle both Pydantic instances and raw dictionaries
        if isinstance(action_effects_data, ActionEffects):
             effects = action_effects_data.effects # Get the nested effects dictionary
        elif isinstance(action_effects_data, dict):
             # Attempt to treat as a dictionary containing the 'effects' key (Pydantic v2 input format)
             # Or a simple dictionary if Pydantic v1 __root__ format was used
             if "effects" in action_effects_data and isinstance(action_effects_data["effects"], dict):
                  effects = action_effects_data["effects"] # Pydantic v2 input format
             else:
                  # Assume it's a simple dictionary of effects (Pydantic v1 __root__ format or raw dict)
                  effects = action_effects_data
        # else:
             # print(f"Warning: Action effects for '{action_key}' is of unexpected type {type(action_effects_data)}. No effects applied.") # Suppress warnings
             # effects remains empty


        # print(f"Applying effects for action '{action_key}' (index {action}): {effects}") # Suppress frequent prints

        # Iterate through the variables and their corresponding delta changes for this action
        for variable, delta in effects.items():
            found = False
            # Find which module instance has this state variable and update its value
            for module_name, module_instance in self.module_instances.items():
                if hasattr(module_instance, variable):
                    current_value = getattr(module_instance, variable)
                    # Update the state variable, clipping the value between 0.0 and 1.0
                    updated_value = np.clip(current_value + delta, 0.0, 1.0)
                    setattr(module_instance, variable, updated_value)
                    # print(f"  Updated '{variable}' in '{module_name}' by {delta:+.2f}. New value: {updated_value:.4f}") # Suppress frequent prints
                    found = True
                    break # Found the variable, move to the next effect
            if not found:
                 # print(f"  Warning: Action effect for variable '{variable}' not applied. Variable not found in any module instances.") # Suppress frequent warnings
                 pass # Silently skip if not found

## Part 3: Data Preprocessor

The `DataPreprocessor` class is a crucial component of the RL training pipeline. Its role is to bridge the gap between the raw usage logs generated by the thesis assistant and the structured input required by the Reinforcement Learning environment and agent.

**Purpose:**
The main purpose is to transform detailed log entries, which capture various events and state information from a user's interaction with the assistant, into a standardized numerical state vector that the RL agent can observe and a corresponding reward signal based on the events in the log.

**Key Components:**
- `__init__(config)`: Initializes the preprocessor with the current RL configuration, which includes definitions of state variables and reward mapping.
- `extract_state(log_entry)`: Takes a single log entry (a dictionary) and converts it into a normalized NumPy array representing the state vector. It selects the relevant information from the log entry based on the `state_variables` defined in the configuration.
- `compute_reward(log_entry)`: Calculates the reward associated with a log entry. It looks for specific event keys within the log entry (as defined in the `reward_config` in the configuration) and sums up the corresponding reward values.

**How to Use:**
- Create an instance of `DataPreprocessor`, passing the loaded RL configuration: `preprocessor = DataPreprocessor(config)`.
- For each raw usage log entry (a dictionary), call `preprocessor.extract_state(log_entry)` to get the state vector and `preprocessor.compute_reward(log_entry)` to get the reward.

**Interaction with Other Components:**
- **Configuration Manager:** The `DataPreprocessor` relies heavily on the configuration loaded by `RLConfigManager` to know which log attributes correspond to state variables and how to map events to reward values.
- **RL Training Loop:** The `RLTrainingLoop` uses the `DataPreprocessor` to process batches of logs before feeding the resulting state and reward information (or using it to update the environment's state and compute rewards in a more interactive simulation) to the RL agent for training.
- **Simulators (Synthetic Cohort/Student):** The simulator classes generate log entries that are in a format expected by the `DataPreprocessor`.

**Data Structures and Configuration:**
- It processes input in the form of dictionaries (log entries).
- It outputs a NumPy array for the state and a float for the reward.
- It uses the `state_variables` and `reward_config` from the RL configuration dictionary to perform the conversion and computation. The keys in `reward_config` are expected to potentially appear as boolean flags or other relevant values in the input `log_entry` dictionaries.

In [234]:
# PART 3 — DATA PREPROCESSOR (LOG TO STATE CONVERSION)
# ===========================================================

class DataPreprocessor:
    """
    Converts raw usage logs into RL state vectors and reward labels.

    This class is responsible for transforming the detailed log entries
    from the thesis assistant's usage into a format (state vectors and rewards)
    that the Reinforcement Learning agent can understand and use for training.
    """

    def __init__(self, config):
        """
        Initialize with the current RL configuration structure.

        Args:
            config (dict): Loaded RL configuration dictionary.
        """
        self.config = config
        # Ensure reward_config contains Pydantic RewardItem instances or is a dict of dicts
        self.reward_config = self.config.get("reward_config", {})


    def extract_state(self, log_entry):
        """
        Convert a single log entry into an RL state vector.

        The state vector is constructed based on the state variables defined
        in the loaded configuration. Values are normalized where appropriate.

        Args:
            log_entry (dict): A single usage log entry.

        Returns:
            state (np.ndarray): The normalized RL state vector.
        """
        state = []
        # Iterate through state variables defined in the config
        for var in self.config["state_variables"]:
            if var == "timestep":
                # For timestep, use deadline_ratio from the log entry and normalize (assuming max timestep is 100 for normalization)
                # Note: In a real scenario, you might use a dedicated 'timestep' in the log or infer it differently.
                # Using deadline_ratio as a proxy for progress/time in this simulation.
                # Access the value using .get for safety in case the key is missing
                state.append(log_entry.get("deadline_ratio", 0.0))
            else:
                # For other variables, get the value directly from the log entry (defaulting to 0.0 if not present)
                # Access the value using .get for safety
                value = log_entry.get(var, 0.0)
                # Ensure the value is a float for consistency
                state.append(float(value))
        return np.array(state, dtype=np.float32) # Ensure float32 dtype for compatibility with RL libraries


    def compute_reward(self, log_entry):
        """
        Compute the shaped reward for a given log event.

        The reward is calculated based on the 'reward_config' in the loaded
        configuration and the presence of specific keys (representing events)
        or state variable values in the log entry. This implementation mirrors
        the logic in `SupervisorEnv._compute_reward` but operates on a static
        log entry rather than the live environment state.

        Args:
            log_entry (dict): A single usage log entry.

        Returns:
            reward (float): The computed reward value.
        """
        total_reward = 0.0
        # print("\n--- DataPreprocessor: Computing Reward ---") # Debug print
        # print("Log Entry (sample):", dict(list(log_entry.items())[:5])) # Print sample

        # Iterate through the reward configuration items
        for reward_name, reward_item_data in self.reward_config.items():
            contribution = 0.0
            # Handle both Pydantic instances and raw dictionaries
            if isinstance(reward_item_data, RewardItem):
                 reward_item = reward_item_data
                 variable = reward_item.variable
                 condition_type = reward_item.condition_type
                 value = reward_item.value
                 threshold = reward_item.threshold
                 scaling_factor = reward_item.scaling_factor
            elif isinstance(reward_item_data, dict):
                 # Attempt to treat as a raw dictionary matching RewardItem schema
                 # This is a fallback for when Pydantic validation was bypassed
                 variable = reward_item_data.get("variable")
                 condition_type = reward_item_data.get("condition_type")
                 value = reward_item_data.get("value", 0.0)
                 threshold = reward_item_data.get("threshold", 0.0)
                 scaling_factor = reward_item_data.get("scaling_factor", 1.0)
                 # Check if essential keys are present for a dict
                 if variable is None or condition_type is None:
                      # print(f"    Warning: Reward item '{reward_name}' is a dictionary but missing essential keys (variable, condition_type). Skipping.") # Suppress warnings
                      continue # Skip this item if essential keys are missing
            else:
                 # print(f"    Warning: Reward item '{reward_name}' is of unexpected type {type(reward_item_data)}. Skipping.") # Suppress warnings
                 continue # Skip if not a Pydantic instance or dictionary

            # print(f"  Evaluating reward item '{reward_name}': variable='{variable}', condition='{condition_type}', value={value}, threshold={threshold}, scaling={scaling_factor}") # Debug print

            # Get the source value from the log entry
            source_value = log_entry.get(variable, None) # Use None to differentiate missing from 0.0

            if source_value is None:
                # Variable not found in log entry
                # print(f"    Warning: Reward variable '{variable}' not found in log entry. Contribution is 0.") # Suppress warnings
                continue # Skip this reward item if the variable is not available

            # Calculate contribution based on condition type and source value
            if condition_type == "event_present":
                 # Expected source_value is boolean or can be interpreted as boolean
                 if bool(source_value) is True:
                      contribution = value * scaling_factor
                      # print(f"    Condition 'event_present' met (value={source_value}). Contribution: {contribution:.4f}") # Debug print
            elif condition_type == "threshold_greater":
                 # Expected source_value is numerical
                 try:
                     if float(source_value) > threshold:
                          contribution = value * scaling_factor
                          # print(f"    Condition '{variable} > {threshold}' met ({float(source_value):.4f} > {threshold}). Contribution: {contribution:.4f}") # Debug print
                 except (ValueError, TypeError):
                      # print(f"    Warning: Variable '{variable}' value ({source_value}) is not numerical for threshold_greater check. Skipping.") # Suppress warnings
                      pass # Skip if source value is not numerical
            elif condition_type == "threshold_less":
                 # Expected source_value is numerical
                 try:
                     if float(source_value) < threshold:
                          contribution = value * scaling_factor
                          # print(f"    Condition '{variable} < {threshold}' met ({float(source_value):.4f} < {threshold}). Contribution: {contribution:.4f}") # Debug print
                 except (ValueError, TypeError):
                      # print(f"    Warning: Variable '{variable}' value ({source_value}) is not numerical for threshold_less check. Skipping.") # Suppress warnings
                      pass # Skip if source value is not numerical
            elif condition_type == "linear_scale":
                 # Expected source_value is numerical
                 try:
                     # Reward scales linearly with the variable value
                     contribution = value * float(source_value) * scaling_factor
                     # print(f"    Condition 'linear_scale' applied ({value} * {float(source_value):.4f} * {scaling_factor}). Contribution: {contribution:.4f}") # Debug print
                 except (ValueError, TypeError):
                      # print(f"    Warning: Variable '{variable}' value ({source_value}) is not numerical for linear_scale. Skipping.") # Suppress warnings
                      pass # Skip if source value is not numerical
            # Add other condition types as needed (e.g., "exact_match", "range")


            total_reward += contribution

        # The DataPreprocessor's compute_reward is primarily for processing *historical* logs
        # into state/reward pairs for training, typically in offline or batch settings.
        # The environment's _compute_reward is for calculating rewards during live
        # environment steps, incorporating dynamic state changes and potentially
        # API costs or time penalties which are environment-specific.
        # For consistency with the env's reward calculation, we should also include
        # the API cost and time penalty here if the DataPreprocessor is used
        # to generate rewards for training data that should match the env.
        # However, for simplicity in this simulated batch training scenario,
        # we might only include the reward_config based contributions.
        # Let's stick to reward_config contributions for the DataPreprocessor for now,
        # as it's processing static logs, not live environment steps.
        # The environment's _compute_reward will include the dynamic penalties.

        # Remove the placeholder penalties/bonuses that were here previously
        # reward -= log_entry.get("ethical_flags", 0.0) * 2.0
        # reward += (1.0 - log_entry.get("ai_usage", 0.0)) * 0.5 # Example bonus
        # reward += log_entry.get("advisor_feedback", 0.0) * 1.0 # Example bonus


        # print(f"--- DataPreprocessor: Reward Calculation Complete ---") # Debug print
        # print(f"  Total Reward: {total_reward:.4f}") # Debug print
        # print("----------------------------------------------------") # Debug print

        return total_reward

**Reasoning**:
The previous command failed because I am still incorrectly using `code_block` for markdown content. I will create a markdown cell to document the PPO Supervisor.



## Part 4: PPO Supervisor (PPORL)

The `PPORL` class is responsible for managing the Proximal Policy Optimization (PPO) agent, which serves as the core Reinforcement Learning component of the assistant. This class handles the initialization, training, saving, loading, and action recommendation (inference) for the PPO model.

**Purpose:**
To implement and control the RL agent that learns an optimal policy for intervening in the thesis writing process to promote ethical behavior and positive outcomes. It trains the agent by interacting with the `SupervisorEnv` and provides action recommendations based on the current state.

**Key Components:**
- `__init__(config, model_path="ethics_rl_model")`: Initializes the PPO supervisor. It creates an instance of the `MockEthicsModule` (to represent the system state), initializes the `SupervisorEnv` using the provided configuration, and either loads a pre-trained PPO model from `model_path` or initializes a new PPO model if no saved model is found.
- `train(timesteps=50000)`: Trains the PPO model for a specified number of environment interaction steps. It calls the `learn()` method of the Stable Baselines3 PPO model, which handles the data collection (interacting with the environment), policy optimization, and value function updates. After training, it saves the updated model.
- `recommend_action()`: Takes the current state from the environment, feeds it to the trained PPO model's policy, and returns the recommended action index. This is the method used during online operation to get the agent's decision.

**How to Use:**
- Initialize the supervisor: `rl_supervisor = SupervisorRL(config, model_path="my_model")`. This will either load an existing model or create a new one.
- To train the model, call `rl_supervisor.train(timesteps=100000)`.
- To get an action recommendation based on the current environment state, call `action = rl_supervisor.recommend_action()`.

**Interaction with Other Components:**
- **Configuration Manager:** The supervisor uses the configuration (loaded indirectly via the environment initialization) to define the RL problem (state/action spaces).
- **RL Environment:** The supervisor interacts directly with the `SupervisorEnv` during training (calling `env.step()`) and inference (getting the state via `env._get_state()`).
- **Mock Ethics Module:** The supervisor initializes an instance of `MockEthicsModule` and passes it to the environment. The environment then uses this object to manage the state.
- **RL Training Loop:** The `RLTrainingLoop` uses the `SupervisorRL` instance to perform the actual training (`trainer.train()`) and potentially get action recommendations (`trainer.recommend_action()`) within its training orchestration.
- **Synthetic Pretrainer:** The `SyntheticRLPretrainer` uses the `RLTrainingLoop`, which in turn uses `SupervisorRL`, to train the model on synthetic data.

**Data Structures and Configuration:**
- It manages a `stable_baselines3.PPO` model.
- It interacts with the environment using NumPy arrays for states and integers for actions.
- The PPO model's policy and value function are learned from the experience collected by interacting with the `SupervisorEnv`, which is configured using the dictionary loaded by `RLConfigManager`.


===========================================================

APPENDIX — PPO SUITABILITY ANALYSIS FOR THESIS ASSISTANT

===========================================================

Analysis: Strengths and Limitations of PPO for Thesis Assistant RL System

✅ PROS (Why PPO is suitable globally):

Stable policy optimization even in high-dimensional state spaces.

Supports multiple complex actions (advisory interventions, ethical warnings, etc).

Optimizes long-term reward (handles delayed ethical consequences).

Clipping mechanism stabilizes policy updates (critical for safe ethical behavior).

Can be trained globally across many users for a general ethical baseline.


⚠ CONS (Limitations for personalization scenario):

Requires many training samples to fully converge (sample inefficient).

Slow adaptation when applied directly to new individual students.

May not personalize fast enough during limited thesis timeframe (6-12 months).

Potential difficulty adapting to individual personality shifts quickly.

PPO only indirectly receives feedback via reward — few-shot adaptation is hard.


✅ RECOMMENDED STRATEGY:

Use PPO for global pretraining across many students (shared ethical policy).

Introduce a lightweight per-student adaptation layer (small fine-tuning component).

Combine PPO with human-in-the-loop reward shaping for rapid personalization.

Consider hybrid architecture with PPO + bandits or meta-RL elements for few-shot adjustments.


This hybrid design balances PPO’s global stability with efficient short-term personalization needs of the thesis assistant. """
"""

In [224]:
# PART 5: PPO SUPERVISOR (Reinforcement Learning Agent Controller)
# ===========================================================
#
# This module manages the PPO RL agent training, saving, loading, and inference.
#
# - Clean separation of agent control logic from environment definition.
# - Compatible with stable-baselines3 PPO implementation.
# - Supports continual training and model persistence.

class SupervisorRL:
    """
    PPO Supervisor class controlling RL training and inference for the Ethics Supervisor.

    This class initializes, trains, saves, and loads the PPO model that acts
     as the RL agent for the ethics module. It interacts with the
    `SupervisorEnv` to learn the optimal intervention policy.

    Attributes:
        module_instances (Dict[str, Any]): Dictionary holding instances of all mock modules.
        env (SupervisorEnv): The RL environment instance.
        model (PPO): The Stable Baselines3 PPO policy model.
        model_path (str): The file path for saving and loading the PPO model.
    """

    def __init__(self, config, model_path="ethics_rl_model"):
        """
        Initialize the PPO agent.

        Args:
            config (dict): The RL configuration dictionary.
            model_path (str): The storage path for saving/loading the PPO model.
                              Defaults to "ethics_rl_model".
        """

        # Initialize instances of all mock modules
        self.module_instances = {
            "ethics": SimpleMockEthicsModule(config),
            "writing": SimpleMockWrittingModule(config),
            "emotion": SimpleMockEmotionModule(config),
            "idea": SimpleMockIdeaModule(config),
            # Add other modules here as they are created
        }
        print("SupervisorRL: Initialized mock module instances:", list(self.module_instances.keys()))

        # Initialize the RL environment with the mock module instances and the provided config
        self.env = SupervisorEnv(self.module_instances, config)
        print("SupervisorRL: Initialized SupervisorEnv with module instances.")

        self.model_path = model_path

        # Load an existing model if available, otherwise initialize a new one
        # Ensure the environment passed to load() has the correct observation space
        if os.path.exists(model_path + ".zip"):
            try:
                # When loading, the environment's observation space must match the saved model's
                # Since self.env was initialized with the current config, this should now match
                self.model = PPO.load(model_path, env=self.env)
                print("SupervisorRL: Loaded pretrained RL model.")
            except ValueError as e:
                print(f"SupervisorRL: Error loading model: {e}. The configuration/environment space may have changed.")
                print("SupervisorRL: Initializing a new model instead.")
                # Initialize a new PPO model if loading fails due to space mismatch
                self.model = PPO("MlpPolicy", self.env, verbose=0) # verbose=0 to suppress training output
        else:
            # Initialize a new PPO model with an MlpPolicy
            self.model = PPO("MlpPolicy", self.env, verbose=0) # verbose=0 to suppress training output
            print("SupervisorRL: Initialized new PPO model.")

    def train(self, timesteps=50000):
        """
        Train the PPO model for a specified number of timesteps.

        The model interacts with the environment to collect experience and update
        its policy.

        Args:
            timesteps (int): The total number of environment steps to train for.
                             Defaults to 50000.
        """
        print(f"SupervisorRL: Starting training for {timesteps} timesteps.")
        self.model.learn(total_timesteps=timesteps)
        self.model.save(self.model_path) # Save the model after training
        print("SupervisorRL: Training complete and model saved.")

    def recommend_action(self):
        """
        Predict the next action based on the current state of the environment.

        This method uses the trained PPO policy to select an action given the
        current observation from the environment.

        Returns:
            int: The action index selected by the PPO policy.
        """
        # Get the current state from the environment
        # Note: In a real application, you would get the real system state
        # and process it into a format matching the environment's observation space.
        state = self.env._get_state()
        # Predict the action using the trained model in deterministic mode
        action, _ = self.model.predict(state, deterministic=True)
        # The action is returned as a NumPy array, so extract the scalar value
        return int(action)

## Part 5: Continual RL Training Loop

The `RLTrainingLoop` class is designed to orchestrate the process of continually training the Reinforcement Learning agent using batches of usage logs. It acts as a bridge between the raw data (logs) and the RL training process managed by the `SupervisorRL`.

**Purpose:**
To facilitate the training of the RL agent using collected usage data. It takes batches of logs, processes them into states and rewards using the `DataPreprocessor`, and then triggers the training of the PPO agent managed by the `SupervisorRL`. This allows for updating the agent's policy as new data becomes available.

**Key Components:**
- `__init__(config, model_path="ppo_ethics_model")`: Initializes the training loop. It loads the configuration, initializes a `DataPreprocessor` instance, and initializes an `SupervisorRL` instance (which handles the PPO model).
- `run_training_day(log_batch)`: The main method for processing a batch of logs. It iterates through each log entry in the `log_batch`, uses the `DataPreprocessor` to extract the state and compute the reward, prints this information, and then calls the `train()` method of the `SupervisorRL` instance to update the PPO model using the processed data. It also demonstrates getting a recommended action after training.

**How to Use:**
- Initialize the training loop: `training_loop = RLTrainingLoop(config)`.
- Provide a batch of usage logs (a list of dictionaries) to the `run_training_day()` method: `training_loop.run_training_day(my_log_batch)`.

**Interaction with Other Components:**
- **Configuration Manager:** The training loop loads the configuration via `RLConfigManager` during its initialization.
- **Data Preprocessor:** It uses the `DataPreprocessor` instance to convert raw log entries into state vectors and reward values.
- **PPO Supervisor (RL Agent):** It uses the `SupervisorRL` instance to perform the actual training of the PPO model using the processed log data.
- **Synthetic Pretrainer and RL Training Launcher:** These classes utilize the `RLTrainingLoop` to execute the training process, providing it with batches of logs (either synthetic or real).

**Data Structures and Configuration:**
- It takes a list of dictionaries (log entries) as input for training.
- It uses the `DataPreprocessor` to work with state vectors (NumPy arrays) and reward values (floats).
- The training process itself is managed by the `SupervisorRL` class, which interacts with the `SupervisorEnv` based on the configuration.

**Reasoning**:
The previous command failed because I am still incorrectly using `code_block` for markdown content. I will create a markdown cell to document the Continual RL Training Loop.



In [225]:
# PART 6 — CONTINUAL RL TRAINING LOOP
# ===========================================================

class RLTrainingLoop:
    """
    Orchestrates the continual RL training process using batches of usage logs.

    This class manages the flow of data from usage logs to the RL training process.
    It uses the `DataPreprocessor` to convert logs into states and rewards and
    the `SupervisorRL` to train the PPO agent.
    """

    def __init__(self, config, model_path="ppo_ethics_model"):
        """
        Initialize training loop components.

        Args:
            config (dict): The loaded RL configuration.
            model_path (str): The path to the PPO model storage. Defaults to "ppo_ethics_model".
        """
        # Corrected call to load_config using the class name
        # Fixed: Instantiate RLConfigManager first, then call load_config on the instance
        config_manager = RLConfigManager()
        self.config = config_manager.load_config() # Load configuration
        self.preprocessor = DataPreprocessor(self.config) # Initialize data preprocessor
        # Initialize the RL trainer, passing the config and model path
        self.trainer = SupervisorRL(self.config, model_path)

    def run_training_day(self, log_batch):
        """
        Process one batch of usage logs and train the PPO agent.

        This method iterates through the provided log batch, processes each log
        into a state and reward, and then uses these to train the RL agent.
        Note: In a true online RL setting, training would occur more frequently
        and interactively with the environment. This simulates batch training
        from collected logs.

        Args:
            log_batch (list of dict): A list of usage logs for one training day or batch.
        """
        print(f"Processing batch of {len(log_batch)} logs for training...")
        # In a real RL loop, you would accumulate experiences (state, action, reward, next_state, done)
        # from interacting with the environment, and then train the agent on these experiences.
        # For this simulated training loop, we will process each log entry and call trainer.train
        # with a small number of timesteps based on the batch size. This is a simplification
        # and not a standard RL training loop, but demonstrates the integration points.
        for log_entry in log_batch:
            state_vector = self.preprocessor.extract_state(log_entry)
            reward = self.preprocessor.compute_reward(log_entry)
            # In a real scenario, you would step the environment with an action and get the next state and reward
            # For this simulated training loop, we'll just print the processed info
            print(f"Processed Log → State: {state_vector}, Reward: {reward}")

        # The training method in SupervisorRL is named 'train'
        # In a real RL loop, you would accumulate experiences and then train
        # For this simulation, we'll call train after processing the batch, using the batch size as timesteps
        # A more realistic approach would involve running episodes in the environment
        # and training on the collected trajectories.
        self.trainer.train(timesteps=len(log_batch)) # Train for the number of logs in the batch
        # The prediction method in SupervisorRL is named 'recommend_action'
        # This is just an example of how to use the trained model after training the batch
        action = self.trainer.recommend_action()
        print(f"Recommended action after training on batch: {action}")

## Part 6: LangGraph Policy Orchestrator

This module defines the policy orchestration layer for the RL-based thesis assistant. It uses **LangGraph** to coordinate four modular support systems:

- Ethics monitoring
- Writing assistance
- Emotional regulation
- Idea generation

The LangGraph agent makes decisions based on the agent’s internal state, current thesis stage, and environment feedback. It invokes modular intervention functions based on valid transitions defined in `rl_config.json`.

This orchestration layer sits **on top of the RLConfigurationManager**, using its `state_variables`, `actions`, `action_effects`, and `action_transitions` to build a dynamic execution graph.

* * *

## Components

- `ThesisState` (in `thesis_modules.py`):
  Typed dictionary defining the agent’s working memory. Includes:
  - `core`: scalar state values (e.g. `advisor_trust`, `creativity_score`)
  - `policy_trace`: list of all actions taken
  - `log`: human-readable trace of module messages
  - `config`: The loaded RL configuration

- `thesis_modules.py`:
  Contains the dummy/stub implementations of the modular action handlers (`ethics_module_actions`, `writing_support_module_actions`, etc.). It exports a combined `__actions__` dictionary containing all available action functions.

- `apply_action_effects` (in `langgraph_policy.py`):
  Decorator function that mutates the state using `action_effects[action_key]` and increments the timestep after each action executes. It also appends the executed action key to the `policy_trace`.

- `route_action` (in `langgraph_policy.py`):
  Function that determines the next action key based on the current state and the `action_transitions` defined in the configuration. In a real RL setting, this would be replaced by a trained policy.

- `build_policy_graph(config)` (in `langgraph_policy.py`):
  Constructs the LangGraph dynamically based on the JSON config. It adds nodes for each action using the functions from `__actions__` (wrapped with `apply_action_effects`), adds an `END` node, and sets up conditional edges using the `route_action` function to define transitions between nodes and from the entry point.

* * *

## Example Actions (from modules)

- `"eth_0"` → Ethics module: Display ethical reminder
- `"eth_1"` → Ethics module: Propose AI restriction
- `"write_0"` → Writing module: Suggest rewriting section
- `"write_1"` → Writing module: Recommend outline reform
- `"emo_0"` → Emotion module: Encourage autonomy
- `"emo_1"` → Emotion module: Acknowledge deadline stress
- `"brain_0"` → Idea module: Prompt open-ended reflection
- `"brain_1"` → Idea module: Offer question inversion


Each action key corresponds to a Python function in `thesis_modules.py` that processes the current state and returns an updated one (before the `apply_action_effects` decorator is applied).

* * *

## Execution Flow

1. Load the RL configuration using `RLConfigManager.load_config()`.
2. Build the LangGraph using `build_policy_graph(config)`, which dynamically adds nodes (actions) and defines transitions based on the loaded config.
3. Invoke the compiled LangGraph with an initial `ThesisState`.
4. The graph starts at the entry point, which calls `route_action` to determine the first action based on the initial state.
5. The chosen action function is executed.
6. The `apply_action_effects` decorator automatically updates the core state based on `action_effects` and appends the action to the `policy_trace`.
7. After the action node completes, the graph transitions back to the routing logic (implicitly via conditional edges from the action node), which calls `route_action` again to determine the next step based on the updated state and the `action_transitions` in the config.
8. This loop continues until `route_action` returns "END".
9. The final state, including the complete policy trace and log, is returned.
**Interaction with Other Components:**
- **Configuration Manager:** The `DeveloperDashboard` directly interacts with `RLConfigManager` to load the initial configuration when launched and to save the updated configuration.
- **RL System Components:** The changes made through the dashboard directly influence how the `DataPreprocessor`, `SupervisorEnv`, and the RL agent (`SupervisorRL`) behave when they load the updated configuration.

**Data Structures and Configuration:**
The dashboard works directly with the dictionary structure managed by `RLConfigManager`. Changes made in the UI are reflected in the `self.config` dictionary within the `DeveloperDashboard` instance before being saved.

In [238]:
# PART 3: LangGraph Policy Orchestrator

from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, END
import numpy as np
import random # Import random for initialization

# Re-define ThesisState to be compatible with the dynamic environment and include env instance and episode status
class ThesisState(TypedDict):
    """
    Full state definition for the LangGraph.

    Combines core state variables (dynamically collected from modules),
    policy execution trace, logs, the configuration dictionary,
    the environment instance, and episode termination status.
    """
    core: Dict[str, float] # Flexible dictionary to hold dynamic state variables
    policy_trace: List[str] # List of action keys executed in sequence
    log: List[str] # Log of messages or events generated by actions
    config: Dict[str, Any] # The loaded RL configuration
    env: Any # Hold the SupervisorEnv instance (using Any as it's defined later)
    done: bool # Flag indicating if the episode is finished
    truncated: bool # Flag indicating if the episode was truncated
    next_node: str # Used by the RL policy node to indicate the next node


# Assuming __actions__ is defined in the mock modules cell and imported or available in the global scope
# from thesis_modules import __actions__ # Uncomment if in separate file

# Assuming SupervisorEnv is defined in a previous cell and available
# from rl_environment import SupervisorEnv # Uncomment if in separate file

# Assuming RLConfigManager is defined in a previous cell and available
# from rl_config_manager import RLConfigManager # Uncomment if in separate file

def apply_action_effects(action_key: str):
    """
    Decorator function to apply action effects to the state using the environment's step method.

    This decorator wraps an action function. After the original action function
    executes and potentially updates the state (e.g., logs a message), this
    decorator uses the environment instance stored in the state to take a step
    with the chosen action. This allows the environment to apply the numerical
    changes to the state variables across the various module instances and
    update the timestep and episode status.

    Args:
        action_key (str): The key of the action being executed (e.g., "eth_0").

    Returns:
        Callable: The decorated action function.
    """
    def decorator(fn):
        def wrapped(state: ThesisState) -> ThesisState:
            print(f"\n--- Executing Action Node: {action_key} ---")
            print(f"Initial state in {action_key} node (sample core):", dict(list(state.get("core", {}).items())[:5]))
            print(f"Initial timestep in {action_key} node:", state.get("core", {}).get("timestep", "N/A"))


            # Execute the original action function (e.g., adds to log)
            print(f"Calling original function for action: {action_key}")
            state = fn(state)
            print(f"Original function for {action_key} executed. Log after fn: {state.get('log', [])[-1] if state.get('log') else 'Empty'}")


            # Append the action to the policy trace *before* stepping the environment
            # so the trace reflects the action taken at the start of the step.
            state["policy_trace"].append(action_key)
            print(f"Appended '{action_key}' to policy trace. Trace length: {len(state['policy_trace'])}")


            # Use the environment instance stored in the state to take a step.
            env = state["env"]
            # Need to convert the action_key back to an action index for the environment's step method
            actions_list = list(state["config"].get("actions", {}).keys())
            try:
                action_index = actions_list.index(action_key)
                print(f"Converted action key '{action_key}' to index: {action_index}")
            except ValueError:
                print(f"Warning: Action key '{action_key}' not found in config actions. Cannot step environment. Returning current state.")
                # Return current state without stepping the environment if action key is invalid
                state["next_node"] = END # Indicate termination due to invalid action key
                return state


            # Take a step in the environment
            print(f"Calling env.step({action_index})...")
            next_obs_array, reward, done, truncated, info = env.step(action_index)
            print(f"env.step() returned. Reward: {reward:.4f}, Done: {done}, Truncated: {truncated}")


            # Update the state dictionary with the results from the environment step
            # Convert the numpy observation array back into the core state dictionary format
            state_variables_order = state["config"].get("state_variables", [])
            next_core_state = {}
            for i, var_name in enumerate(state_variables_order):
                 if i < len(next_obs_array): # Ensure index is within bounds
                      next_core_state[var_name] = float(next_obs_array[i]) # Convert numpy float to standard float
                 else:
                      # print(f"Warning: Observation array index out of bounds for variable '{var_name}'.") # Suppress warnings
                      next_core_state[var_name] = 0.0 # Default to 0.0 if obs size mismatch


            state["core"] = next_core_state
            state["done"] = done
            state["truncated"] = truncated
            # Optionally add reward and other info to the log or a separate state field
            state["log"].append(f"Env step taken with action '{action_key}'. Reward: {reward:.4f}, Done: {done}, Truncated: {truncated}. Info: {info.get('simulated_log_entry', 'N/A')}") # Include simulated log

            print(f"State updated after env.step(). New timestep: {state.get('core', {}).get('timestep', 'N/A')}")
            print(f"Exiting action node {action_key}. Returning updated state.")

            # This node doesn't set next_node; the edge from this node determines the next step (back to rl_policy_node)
            return state
        return wrapped
    return decorator


# Define the new node function for running the RL policy
def run_rl_policy_node(state: ThesisState) -> ThesisState:
    """
    LangGraph node function to get an action recommendation from the RL agent
    and update the state to indicate the next node.
    """
    print("\n--- Running RL Policy Node ---")
    print("Received state (sample core):", dict(list(state.get("core", {}).items())[:5])) # Print sample keys
    print("Current timestep:", state.get("core", {}).get("timestep", "N/A"))


    # Check if episode is done or truncated - route to END
    if state.get("done", False) or state.get("truncated", False):
        print("RL Policy Node: Episode terminated by environment. Setting next_node to END.")
        state["next_node"] = END # Indicate termination
        return state


    # 1. Access the SupervisorEnv instance from state
    env = state.get("env")
    if env is None:
        print("Error: SupervisorEnv instance not found in state. Cannot run RL policy. Setting next_node to END.")
        state["next_node"] = END
        return state

    # 2. Get the current observation/state vector from the env using env._get_state()
    try:
        observation = env._get_state()
        print("Got observation from environment (sample):", observation[:min(5, len(observation))]) # Print sample values
        print("Observation shape:", observation.shape)

    except Exception as e:
        print(f"Error getting observation from environment: {e}. Cannot run RL policy. Setting next_node to END.")
        state["next_node"] = END
        return state


    # 3. Access the SupervisorRL agent instance (assuming global access for now)
    global rl_supervisor # Declare global to access the instance defined outside this function
    print(f"RL Policy Node: Attempting to access global rl_supervisor. Is None: {rl_supervisor is None}")
    if 'rl_supervisor' not in globals() or rl_supervisor is None:
         print("Error: SupervisorRL instance 'rl_supervisor' not found in global scope. Cannot run RL policy. Setting next_node to END.")
         state["next_node"] = END
         return state


    # 4. Call the recommend_action() method of the rl_supervisor instance
    try:
        # The recommend_action method already takes the state internally from the env
        # based on its initialization. So we don't pass the observation explicitly here.
        recommended_action_index = rl_supervisor.recommend_action()
        print(f"RL Supervisor recommended action index: {recommended_action_index}")

    except Exception as e:
        print(f"Error getting action recommendation from RL Supervisor: {e}. Cannot run RL policy. Setting next_node to END.")
        state["next_node"] = END
        return state


    # 5. Convert the action index back to its corresponding action key (string)
    config = state.get("config", {})
    actions_dict = config.get("actions", {})
    actions_list = list(actions_dict.keys())

    if recommended_action_index < 0 or recommended_action_index >= len(actions_list):
        print(f"Error: Recommended action index {recommended_action_index} is out of bounds (action space size {len(actions_list)}). Setting next_node to END.")
        state["next_node"] = END
        return state


    recommended_action_key = actions_list[recommended_action_index]
    print(f"Recommended action key: '{recommended_action_key}'")

    # 6. Update the state to indicate the next node
    state["next_node"] = recommended_action_key
    print(f"RL Policy Node: Setting next_node to '{recommended_action_key}'.")


    # Return the updated state
    return state


# Assuming the rest of the LangGraph code (__actions__, etc.)
# and the mock module classes and SupervisorEnv are defined in previous cells.

# Assuming __actions__ is defined in the mock module cell.
# Define a placeholder __actions__ dictionary if it's not guaranteed to be defined yet.
# This is a defensive measure; the actual mock module cell should define this.
if '__actions__' not in globals():
    __actions__ = {
        "dummy_action_1": lambda state: state,
        "dummy_action_2": lambda state: state,
        # Add other placeholder actions if needed for graph building without mocks
    }
    print("Warning: __actions__ not found, using placeholder actions for graph building.")


def build_policy_graph_with_rl(config: Dict[str, Any]) -> StateGraph:
    """
    Builds the LangGraph policy including the RL policy node.

    This function is similar to build_policy_graph but includes the 'run_rl_policy_node'.
    """
    graph = StateGraph(ThesisState)

    # Get action definitions from the config
    actions_config = config.get("actions", {})
    action_effects_config = config.get("action_effects", {}) # Get the effects config


    # Add nodes for each action defined in the configuration's 'actions' dictionary
    # Lookup the actual function from the global __actions__ dictionary
    for action_key, action_label in actions_config.items():
        action_fn = __actions__.get(action_key)
        if action_fn is not None:
             # Wrap the action function with the apply_action_effects decorator
             # Pass the action_key to the decorator
             decorated_action_fn = apply_action_effects(action_key)(action_fn)
             graph.add_node(action_key, decorated_action_fn)
             print(f"Added node for action: {action_key} ('{action_label}')")
        else:
             print(f"Warning: Action key '{action_key}' found in config but no corresponding function in __actions__. Skipping node.")


    # Add the special RL policy node
    graph.add_node("rl_policy_node", run_rl_policy_node) # Use the updated node function
    print("Added RL policy node: rl_policy_node")

    # Add the END node
    graph.add_node("END", lambda state: state)
    print("Added END node.")


    # Add edges:
    # 1. From each action node, route back to the RL policy node to get the next action recommendation.
    for action_key in actions_config.keys():
         # Only add edge if the action node was successfully added to the graph
         if action_key in graph.nodes:
              graph.add_edge(action_key, "rl_policy_node") # After an action, go back to the RL policy node
              # print(f"Added edge from {action_key} to rl_policy_node")


    # 2. From the RL policy node, route conditionally based on the 'next_node' key in the state.
    #    The run_rl_policy_node function sets state["next_node"] to the recommended action key or "END".
    graph.add_conditional_edges(
        "rl_policy_node", # The node to route FROM (the RL policy node)
        lambda state: state.get("next_node", END), # Function to get the next node name from the state
        # No mapping needed here, as the lambda directly returns the target node name.
    )
    print("Added conditional edge from rl_policy_node based on state['next_node'].")


    return graph.compile()


# The original route_action function is not used in this structure anymore.
# It can be kept if used elsewhere, or removed.


# Re-run the main demo block with the updated build_policy_graph_with_rl
if __name__ == "__main__":
    print("--- LangGraph Policy Demo with SupervisorRL Integration (Revised) ---")

    # Assuming RLConfigManager is available (from the previous cell)
    # from rl_config_manager import RLConfigManager

    # Assuming SupervisorEnv and mock module classes are available
    # from rl_environment import SupervisorEnv
    # from mock_modules import SimpleMockEthicsModule, SimpleMockWrittingModule, SimpleMockEmotionModule, SimpleMockIdeaModule
    # Assuming SupervisorRL is available

    # 1. Load configuration using the RLConfigManager
    try:
        mgr = RLConfigManager()
        config = mgr.load_config()
        print("✅ Configuration loaded.")

        # Ensure action_transitions exist for the demo, add if not (still needed for action_effects decorator lookup)
        # Note: action_transitions are NOT used by the RL policy, they are for graph structure in non-RL modes.
        # However, the decorator doesn't strictly need them, but the build process might expect them.
        # Let's ensure they are present in the config if not loaded.
        if "action_transitions" not in config:
             config["action_transitions"] = {
                 "eth_0": {"next": "write_0"},
                 "write_0": {"next": "emo_0"},
                 "emo_0": {"next": "brain_0"},
                 "brain_0": {"next": "END"}
             }
             print("ℹ️ Added default action_transitions for demo (for graph structure).")


        # Ensure state_variables are correctly populated from action effects for the demo
        # This is important for the environment's observation space and the state vector
        vars_from_effects = set()
        if "action_effects" in config:
            # action_effects is now a dict of ActionEffects instances
            for action_effects_instance in config["action_effects"].values():
                if isinstance(action_effects_instance, ActionEffects):
                     vars_from_effects.update(action_effects_instance.effects.keys())
                elif isinstance(action_effects_instance, dict) and "effects" in action_effects_instance and isinstance(action_effects_instance["effects"], dict):
                     vars_from_effects.update(action_effects_instance["effects"].keys())
                elif isinstance(action_effects_instance, dict): # Handle simple dict fallback
                     vars_from_effects.update(action_effects_instance.keys())


        required_state_vars = set(config.get("state_variables", []))
        missing_vars_in_state_vars = list(vars_from_effects - required_state_vars - set(["timestep"]))
        if missing_vars_in_state_vars:
            print(f"  Adding missing variables from action_effects to state_variables in config for demo: {missing_vars_in_state_vars}")
            config.setdefault("state_variables", []).extend(missing_vars_in_state_vars)
            config["state_variables"] = list(dict.fromkeys(config["state_variables"])) # Ensure uniqueness


    except Exception as e:
        print(f"❌ Failed to load configuration: {e}")
        # Fallback to a basic default config if loading fails
        # Ensure fallback config matches the structure expected by modules and env
        config = {
            "config_version": 1.0,
            "created_at": datetime.datetime.now().isoformat(), # Use ISO format for JSON compatibility
            "state_variables": ["timestep", "dummy_var_1"], # Add a dummy var
            "actions": {
                "dummy_action_1": "A first dummy action",
                "dummy_action_2": "A second dummy action",
            },
            "action_effects": {
                "dummy_action_1": {"effects": {"dummy_var_1": 0.1, "timestep": 0.1}}, # Add simple effects
                "dummy_action_2": {"effects": {"dummy_var_1": -0.1, "timestep": 0.1}}, # Add simple effects
            },
            "action_transitions": {
                "dummy_action_1": {"next": "dummy_action_2"},
                "dummy_action_2": {"next": "END"},
            },
            "reward_config": {}, # Empty reward config for fallback
             "ethics_threshold": 0.7, # Add module specific param
        }
        print("⚠️ Using fallback configuration.")


    # Initialize mock module instances (required by SupervisorEnv and SupervisorRL)
    # These instances will hold the state variables
    # Need to ensure that the fallback config includes all necessary state variables
    # expected by the mock module initializers if they access attributes directly.
    # Let's update the mock module initializers to safely access config values.
    # (Already done in the previous fix for SimpleMockEthicsModule)

    mock_module_instances = {
        "ethics": SimpleMockEthicsModule(config),
        "writing": SimpleMockWrittingModule(config),
        "emotion": SimpleMockEmotionModule(config),
        "idea": SimpleMockIdeaModule(config),
    }
    print("\nInitialized mock module instances for LangGraph demo:", list(mock_module_instances.keys()))

    # 2. Initialize the SupervisorRL agent
    # This also initializes the SupervisorEnv internally with the mock module instances
    try:
        # Make rl_supervisor globally accessible for the run_rl_policy node (for this demo)
        global rl_supervisor
        # Check if it was already initialized by the launcher or pretrainer
        if 'rl_supervisor' not in globals() or rl_supervisor is None:
             # Only initialize if not already present
             rl_supervisor = SupervisorRL(config)
             print("✅ SupervisorRL initialized for LangGraph demo.")
        else:
             print("✅ SupervisorRL already initialized. Reusing existing instance.")

    except NameError:
        print("❌ SupervisorRL class not found. Cannot run demo.")
        rl_supervisor = None


    # Define the initial state for the simulation
    # The core state will be populated by the environment's reset method
    initial_state: ThesisState = {
        "core": {}, # Start with an empty core state, will be populated by env.reset()
        "policy_trace": [], # Start with an empty trace
        "log": [], # Start with an empty log
        "config": config, # Pass the loaded config to the state
        "env": None, # Env will be initialized by SupervisorRL, need to get it from there
        "done": False, # Initial episode status
        "truncated": False, # Initial episode status
        # No need for "next_node" in the initial state, as the entry point is set directly
    }

    # Get the environment instance from the initialized SupervisorRL
    env = None
    if rl_supervisor:
         env = rl_supervisor.env
         initial_state["env"] = env

         # Before running the graph, reset the environment
         print("\nResetting environment before running LangGraph...")
         initial_obs, initial_info = env.reset()
         # Populate the initial core state from the environment's reset output
         state_variables_order = config.get("state_variables", [])
         initial_core_state = {}
         # Ensure the number of variables in config matches the observation space from env.reset()
         if len(state_variables_order) != len(initial_obs):
             print(f"Error: Number of state variables in config ({len(state_variables_order)}) does not match observation space size from env.reset() ({len(initial_obs)}).")
             # Attempt to proceed but log the error
             # Use min length to avoid index errors
             num_vars_to_map = min(len(state_variables_order), len(initial_obs))
             for i in range(num_vars_to_map):
                 initial_core_state[state_variables_order[i]] = float(initial_obs[i])
         else:
             for i, var_name in enumerate(state_variables_order):
                  initial_core_state[var_name] = float(initial_obs[i]) # Convert numpy float to standard float


         initial_state["core"] = initial_core_state
         initial_state["done"] = False # Ensure done flag is False after reset
         initial_state["truncated"] = False # Ensure truncated flag is False after reset


         print("Initial core state populated from env.reset():", dict(list(initial_state["core"].items())[:min(5, len(initial_state["core"]))])) # Print sample
         print("Initial env done flag:", initial_state["done"])
         print("Initial env truncated flag:", initial_state["truncated"])
         print("Initial env timestep:", env.timestep) # Should be 0 after reset
    else:
        # If SupervisorRL failed to initialize, cannot proceed
        print("Cannot run LangGraph demo without a valid SupervisorRL.")


    # Build and run the LangGraph
    thesis_graph = None
    if rl_supervisor and env: # Only proceed if SupervisorRL and env were initialized successfully
        try:
            # Use the new build function that includes the RL node
            thesis_graph = build_policy_graph_with_rl(config)
            print("\n✅ LangGraph built with RL policy node.")

            # Invoke the graph to run the policy
            # The graph will start at the entry point ('rl_policy_node')
            print("\nRunning LangGraph...")
            # The graph will continue until the RL policy node returns "END"
            # or the environment signals done/truncated via the apply_action_effects decorator
            # The max_steps argument limits the number of steps in the graph traversal,
            # preventing infinite loops in case of unexpected routing.
            # Set a reasonable limit for the demo.
            max_graph_steps = 10 # Limit the number of graph steps for the demo

            # LangGraph's invoke method returns the final state after execution finishes.
            # Execution finishes when a node returns END or max_steps is reached.
            # Let's invoke and capture the final state.
            # Use stream for step-by-step output
            for step_output in thesis_graph.stream(initial_state, {"recursion_limit": max_graph_steps}):
                 # The stream method yields the state after each node execution.
                 # This allows us to see the state changes step-by-step.
                 for key, value in step_output.items():
                     print(f"\n--- State after '{key}' node ---")
                     # Print relevant parts of the state
                     print("Core (sample):", dict(list(value.get("core", {}).items())[:min(5, len(value.get("core", {})))])) # Print sample key-value pairs
                     print("Timestep:", value.get("core", {}).get("timestep", "N/A"))
                     print("Policy Trace:", value.get("policy_trace", [])[-5:]) # Last few trace items
                     print("Log (last entry):", value.get("log", [])[-1] if value.get("log") else "Empty")
                     print("Done:", value.get("done", False))
                     print("Truncated:", value.get("truncated", False))
                     # Check the next_node if it exists (set by RL node)
                     if "next_node" in value:
                         print("Next Node (set by RL node):", value["next_node"])

                     # Check for episode termination after each step
                     if value.get("done", False) or value.get("truncated", False):
                          print("\nEpisode terminated during stream.")
                          break # Exit the loop if the episode is done


            print("\n✅ LangGraph execution finished (stream complete or limit reached).")


        except Exception as e:
            print(f"\n❌ An error occurred during LangGraph execution: {e}")


    print("\n--- Demo Complete ---")

--- LangGraph Policy Demo with SupervisorRL Integration (Revised) ---
No config source provided, using default configuration.
Using default configuration.
✅ Configuration validated and loaded successfully (retaining Pydantic instances).
✅ Configuration loaded.

Initialized mock module instances for LangGraph demo: ['ethics', 'writing', 'emotion', 'idea']
✅ SupervisorRL already initialized. Reusing existing instance.

Resetting environment before running LangGraph...
Error: Number of state variables in config (16) does not match observation space size from env.reset() (1).
Initial core state populated from env.reset(): {'embedding_drift': 0.0}
Initial env done flag: False
Initial env truncated flag: False
Initial env timestep: 0
Added node for action: eth_0 ('Display ethical reminder')
Added node for action: eth_1 ('Propose AI restriction')
Added node for action: eth_2 ('Recommend advisor check-in')
Added node for action: eth_3 ('Log academic concern')
Added node for action: brain_0 ('P

In [227]:
# Test the LangGraph with the integrated SupervisorRL and multi-module environment

print("\n--- Testing LangGraph with SupervisorRL and Multi-Module Env ---")

# 1. Load configuration using the RLConfigManager
try:
    mgr = RLConfigManager()
    config = mgr.load_config()
    print("✅ Configuration loaded.")

    # Ensure action_transitions exist for the demo, add if not
    # These are needed for the graph structure and the apply_action_effects decorator lookup
    if "action_transitions" not in config:
         config["action_transitions"] = {
             "eth_0": {"next": "write_0"},
             "write_0": {"next": "emo_0"},
             "emo_0": {"next": "brain_0"},
             "brain_0": {"next": "END"}
         }
         print("ℹ️ Added default action_transitions for demo.")


    # Ensure state_variables are correctly populated from action effects for the demo
    # This is important for the environment's observation space and the state vector
    vars_from_effects = set()
    if "action_effects" in config:
        for effects in config["action_effects"].values():
            vars_from_effects.update(effects.keys())
    required_state_vars = set(config.get("state_variables", []))
    missing_vars_in_state_vars = list(vars_from_effects - required_state_vars - set(["timestep"]))
    if missing_vars_in_state_vars:
        print(f"  Adding missing variables from action_effects to state_variables in config for demo: {missing_vars_in_state_vars}")
        config.setdefault("state_variables", []).extend(missing_vars_in_state_vars)
        config["state_variables"] = list(dict.fromkeys(config["state_variables"])) # Ensure uniqueness


except Exception as e:
    print(f"❌ Failed to load configuration: {e}")
    # Fallback to a basic default config if loading fails
    config = {
        "actions": {
            "dummy_action_1": "A first dummy action",
            "dummy_action_2": "A second dummy action",
        },
        "action_effects": {
            "dummy_action_1": {"timestep": 0.1}, # Add a simple effect
            "dummy_action_2": {"timestep": 0.1}, # Add a simple effect
        },
        "action_transitions": {
            "dummy_action_1": {"next": "dummy_action_2"},
            "dummy_action_2": {"next": "END"},
        },
        "state_variables": ["timestep"], # Minimal state variables
         "reward_config": {},
    }
    print("⚠️ Using fallback configuration.")


# 2. Initialize mock module instances (required by SupervisorEnv and SupervisorRL)
# These instances will hold the state variables
mock_module_instances = {
    "ethics": SimpleMockEthicsModule(config),
    "writing": SimpleMockWrittingModule(config),
    "emotion": SimpleMockEmotionModule(config),
    "idea": SimpleMockIdeaModule(config),
}
print("\nInitialized mock module instances for LangGraph test:", list(mock_module_instances.keys()))

# 3. Initialize the SupervisorRL agent
# This also initializes the SupervisorEnv internally with the mock module instances
try:
    # Make rl_supervisor globally accessible for the run_rl_policy node (as done in the main block)
    global rl_supervisor
    rl_supervisor = SupervisorRL(config)
    print("✅ SupervisorRL initialized for LangGraph test.")
except NameError:
    print("❌ SupervisorRL class not found. Cannot run test.")
    rl_supervisor = None


# 4. Define the initial state for the simulation
# The core state will be populated by the environment's reset method
initial_state: ThesisState = {
    "core": {}, # Start with an empty core state, will be populated by env.reset()
    "policy_trace": [], # Start with an empty trace
    "log": [], # Start with an empty log
    "config": config, # Pass the loaded config to the state
    "env": None, # Env will be initialized by SupervisorRL, need to get it from there
    "done": False, # Initial episode status
    "truncated": False, # Initial episode status
    # No need for "next_node" in the initial state, as the entry point is set directly
}

# 5. Get the environment instance from the initialized SupervisorRL and reset it
env = None
if rl_supervisor:
     env = rl_supervisor.env
     initial_state["env"] = env

     # Reset the environment to get the initial observation and populate the core state
     print("\nResetting environment before running LangGraph...")
     initial_obs, initial_info = env.reset()
     # Populate the initial core state from the environment's reset output
     state_variables_order = config.get("state_variables", [])
     initial_core_state = {}
     # Ensure the number of variables in config matches the observation space from env.reset()
     if len(state_variables_order) != len(initial_obs):
         print(f"Error: Number of state variables in config ({len(state_variables_order)}) does not match observation space size from env.reset() ({len(initial_obs)}).")
         # Attempt to proceed but log the error
         # Use min length to avoid index errors
         num_vars_to_map = min(len(state_variables_order), len(initial_obs))
         for i in range(num_vars_to_map):
             initial_core_state[state_variables_order[i]] = float(initial_obs[i])
     else:
         for i, var_name in enumerate(state_variables_order):
              initial_core_state[var_name] = float(initial_obs[i]) # Convert numpy float to standard float


     initial_state["core"] = initial_core_state
     initial_state["done"] = False # Ensure done flag is False after reset
     initial_state["truncated"] = False # Ensure truncated flag is False after reset


     print("Initial core state populated from env.reset():", dict(list(initial_state["core"].items())[:5])) # Print sample
     print("Initial env done flag:", initial_state["done"])
     print("Initial env truncated flag:", initial_state["truncated"])
     print("Initial env timestep:", env.timestep) # Should be 0 after reset
else:
    print("Cannot run LangGraph test without a valid SupervisorRL.")


# 6. Build the LangGraph with the RL policy node
thesis_graph = None
if rl_supervisor and env: # Only proceed if SupervisorRL and env were initialized successfully
    try:
        # Use the new build function that includes the RL node
        thesis_graph = build_policy_graph_with_rl(config)
        print("\n✅ LangGraph built with RL policy node.")

    except Exception as e:
        print(f"\n❌ An error occurred during LangGraph building: {e}")


# 7. Run the graph for a few steps using stream
if thesis_graph:
    print("\nRunning LangGraph stream for a few steps...")
    # The graph will start at the entry point ('rl_policy_node')
    # The graph will continue until the RL policy node returns "END"
    # or the environment signals done/truncated via the apply_action_effects decorator
    # The max_steps argument limits the number of steps in the graph traversal,
    # preventing infinite loops in case of unexpected routing.
    # Set a reasonable limit for the demo to observe a few action/policy cycles.
    max_graph_steps = 10 # Limit the number of graph steps for the test

    try:
        # Stream the graph execution
        for step_output in thesis_graph.stream(initial_state, {"recursion_limit": max_graph_steps}):
             # The stream method yields the state after each node execution.
             # This allows us to see the state changes step-by-step.
             for key, value in step_output.items():
                 print(f"\n--- State after '{key}' node ---")
                 # Print relevant parts of the state
                 print("Core (sample):", dict(list(value.get("core", {}).items())[:5])) # Print sample key-value pairs
                 print("Timestep:", value.get("core", {}).get("timestep", "N/A"))
                 print("Policy Trace:", value.get("policy_trace", [])[-5:]) # Last few trace items
                 print("Log (last entry):", value.get("log", [])[-1] if value.get("log") else "Empty")
                 print("Done:", value.get("done", False))
                 print("Truncated:", value.get("truncated", False))
                 # Check the next_node if it exists (set by RL node)
                 if "next_node" in value:
                     print("Next Node (set by RL node):", value["next_node"])

                 # Check for episode termination after each step
                 if value.get("done", False) or value.get("truncated", False):
                      print("\nEpisode terminated during stream.")
                      break # Exit the loop if the episode is done


        print("\n✅ LangGraph stream finished (complete or limit reached).")

    except Exception as e:
        print(f"\n❌ An error occurred during LangGraph stream execution: {e}")


print("\n--- LangGraph Test Complete ---")


--- Testing LangGraph with SupervisorRL and Multi-Module Env ---
No config source provided, using default configuration.
Using default configuration.
✅ Configuration validated and loaded successfully (retaining Pydantic instances).
✅ Configuration loaded.
❌ Failed to load configuration: 'ActionEffects' object has no attribute 'keys'
⚠️ Using fallback configuration.

Initialized mock module instances for LangGraph test: ['ethics', 'writing', 'emotion', 'idea']
SupervisorRL: Initialized mock module instances: ['ethics', 'writing', 'emotion', 'idea']

--- SupervisorEnv Initialization Debug ---
Received config keys: ['actions', 'action_effects', 'action_transitions', 'state_variables', 'reward_config']
Config type: <class 'dict'>
Type of config['action_effects']: <class 'dict'>
Sample action_effects keys: ['dummy_action_1', 'dummy_action_2']
Type of sample action_effects value ('dummy_action_1'): <class 'dict'>
Sample action_effects value: {'timestep': 0.1}
Type of config['reward_config']

## Part 7: Synthetic Thesis Student Simulator (ThesisStudentSimulator)

The `ThesisStudentSimulator` class provides a way to generate synthetic data that mimics the progression and ethical interactions of a thesis student using the assistant. This simulator is essential for generating datasets to pre-train and test the Reinforcement Learning ethics supervisor in a controlled environment.

**Purpose:**
To create realistic (though simplified) sequences of events and state changes that a thesis student might experience over the course of their project. This synthetic data includes simulated metrics like AI usage, ethical flags, advisor feedback, and progress towards the deadline, which are used to train the RL agent.

**Key Components:**
- `__init__(student_type="Stable Performer")`: Initializes the simulator for a specific type of student (e.g., "Conservative", "Aggressive", "Struggling", "Stable Performer"). Each student type has different tendencies influencing how their state variables evolve.
- `evolve_one_step()`: Simulates one step in the student's thesis journey. It updates the state variables based on the student type and progress towards the deadline and generates a dictionary representing a single log entry for this step.
- `grade_final_outcome(last_state)`: A static method that calculates a final grade and reward for a simulated thesis trajectory based on the state of the simulator at the end of the trajectory. This provides a terminal reward signal for the simulation.
- `generate_full_trajectory_with_grading(student_type, trajectory_length=30)`: A static method that runs a full simulation trajectory for a specified student type and length, collecting all log entries and returning the list of logs along with the final grade and terminal reward.

**How to Use:**
- To simulate a single student's journey, create an instance: `student_sim = ThesisStudentSimulator("Struggling")` and repeatedly call `student_sim.evolve_one_step()` to get step-by-step logs.
- To generate a complete trajectory and its grading, call the static method: `logs, grade, reward = ThesisStudentSimulator.generate_full_trajectory_with_grading("Aggressive", trajectory_length=50)`.

**Interaction with Other Components:**
- **Data Preprocessor:** The `evolve_one_step()` method generates log entries in a dictionary format that is compatible with the input expected by the `DataPreprocessor`.
- **Thesis Cohort Simulator:** The `ThesisCohortSimulator` uses the `generate_full_trajectory_with_grading` method to create datasets for multiple students.
- **Synthetic RL Pretrainer:** The `SyntheticRLPretrainer` utilizes the data generated by the simulator (via the `ThesisCohortSimulator`) to train the RL agent.

**Data Structures and Configuration:**
- The simulator maintains internal state variables (e.g., `ai_usage`, `ethical_flags`) as numerical values, typically floats between 0.0 and 1.0.
- It generates output as dictionaries, where each dictionary represents a single usage log entry containing the state variables and simulated event flags.
- The behavior is influenced by the `student_type` string.

In [228]:
# PART 7 — SYNTHETIC THESIS STUDENT SIMULATOR WITH OUTCOME GRADING
# ===========================================================

class ThesisStudentSimulator:
    """
    Simulates the progress and ethical behavior of a synthetic thesis student.

    This class generates synthetic usage logs and simulates changes in state
    variables (like AI usage, ethical flags, etc.) over time, based on
    predefined student types. It also includes a method to grade the final
    outcome of a simulated thesis trajectory.
    """
    def __init__(self, student_type="Stable Performer"):
        """
        Initialize the simulator for a specific type of student.

        Args:
            student_type (str): The type of student to simulate
                                 ("Conservative", "Aggressive", "Struggling",
                                  "Stable Performer"). Defaults to "Stable Performer".
        """
        self.student_type = student_type
        # Initialize state variables
        self.embedding_drift = 0.2
        self.ai_usage = 0.3
        self.ethical_flags = 0.05
        self.advisor_feedback = 0.6
        self.deadline_ratio = 0.0 # Represents progress towards deadline (0.0 to 1.0)

    def evolve_one_step(self):
        """
        Simulate one step of the student's thesis progress and generate a log entry.

        State variables evolve based on the student type and time progression.

        Returns:
            dict: A dictionary representing a single usage log entry with updated state.
        """
        # Simulate progress towards the deadline
        self.deadline_ratio = min(self.deadline_ratio + 0.03, 1.0) # Increment deadline ratio

        # Simulate changes in state variables based on student type
        if self.student_type == "Conservative":
            self.ai_usage += np.random.normal(0.01, 0.02)
            self.ethical_flags += np.random.normal(0.0, 0.01)
            self.advisor_feedback += np.random.normal(0.02, 0.05)
            self.embedding_drift += np.random.normal(0.01, 0.02)
        elif self.student_type == "Aggressive":
            self.ai_usage += np.random.normal(0.05, 0.05)
            self.ethical_flags += np.random.normal(0.02, 0.03)
            self.advisor_feedback += np.random.normal(-0.02, 0.05)
            self.embedding_drift += np.random.normal(0.03, 0.05)
        elif self.student_type == "Struggling":
            self.ai_usage += np.random.normal(0.03, 0.03)
            self.ethical_flags += np.random.normal(0.05, 0.05)
            self.advisor_feedback += np.random.normal(-0.03, 0.05)
            self.embedding_drift += np.random.normal(0.04, 0.05)
        elif self.student_type == "Stable Performer":
            self.ai_usage += np.random.normal(0.02, 0.02)
            self.ethical_flags += np.random.normal(0.01, 0.01)
            self.advisor_feedback += np.random.normal(0.03, 0.04)
            self.embedding_drift += np.random.normal(0.02, 0.02)

        # Increase ethical flags towards the end of the project (simulating pressure)
        if self.deadline_ratio > 0.8:
            self.ethical_flags += 0.02

        # Simulate some convergence towards target values as the deadline approaches
        convergence_factor = self.deadline_ratio
        self.ai_usage += (0.5 - self.ai_usage) * 0.1 * convergence_factor
        self.ethical_flags += (0.1 - self.ethical_flags) * 0.1 * convergence_factor
        self.advisor_feedback += (0.8 - self.advisor_feedback) * 0.1 * convergence_factor
        self.embedding_drift += (0.3 - self.embedding_drift) * 0.05 * convergence_factor

        # Clip state variables to remain within the [0, 1] range
        self.ai_usage = np.clip(self.ai_usage, 0, 1.0)
        self.ethical_flags = np.clip(self.ethical_flags, 0, 1.0)
        self.advisor_feedback = np.clip(self.advisor_feedback, 0, 1.0)
        self.embedding_drift = np.clip(self.embedding_drift, 0, 1.0)

        # Create a log entry with the current state and some simulated events (for reward calculation)
        log_entry = {
            "embedding_drift": self.embedding_drift,
            "ai_usage": self.ai_usage,
            "ethical_flags": self.ethical_flags,
            "advisor_feedback": self.advisor_feedback,
            "deadline_ratio": self.deadline_ratio,
            # Simulate boolean event flags based on probabilities or state
            "user_revised": random.random() < 0.6, # Probability of user revising content
            "ai_violation": random.random() < self.ethical_flags, # Higher ethical flags increase chance of violation
            "advisor_positive": random.random() < self.advisor_feedback, # Higher feedback increases chance of positive advisor event
            "rewrite_accepted": random.random() < 0.7, # Probability of rewrite suggestion being accepted
            "milestone_completed": random.random() < 0.4, # Probability of completing a milestone
            "hallucination_detected": random.random() < 0.1 # Probability of detecting a hallucination
        }
        return log_entry

    @staticmethod
    def grade_final_outcome(last_state):
        """
        Grades the final outcome of a simulated thesis based on the last state.

        This is a simplified grading function for simulation purposes.

        Args:
            last_state (dict): The final state of the simulator after a trajectory.

        Returns:
            tuple: A tuple containing the grade ("Excellent", "Acceptable", "Failed")
                   and a corresponding numerical reward.
        """
        # Calculate penalties and bonuses based on the final state values
        ai_penalty = (last_state["ai_usage"] - 0.5) * 0.5 # Penalty if AI usage is high relative to 0.5
        ethics_penalty = last_state["ethical_flags"] * 1.5 # Penalty for ethical flags
        advisor_bonus = last_state["advisor_feedback"] * 2.0 # Bonus for positive advisor feedback
        embedding_penalty = last_state["embedding_drift"] * 0.3 # Penalty for high embedding drift
        # Calculate total score
        total_score = advisor_bonus - ethics_penalty - ai_penalty - embedding_penalty

        # Assign grade and reward based on the total score
        if total_score > 1.2:
            return "Excellent", 5.0
        elif total_score > 0:
            return "Acceptable", 2.0
        else:
            return "Failed", -5.0

    @staticmethod
    def generate_full_trajectory_with_grading(student_type, trajectory_length=30):
        """
        Generates a full simulated thesis trajectory for a student and grades it.

        Args:
            student_type (str): The type of student to simulate.
            trajectory_length (int): The number of steps in the simulation trajectory.
                                     Defaults to 30.

        Returns:
            tuple: A tuple containing:
                   - logs (list of dict): The list of log entries generated during the trajectory.
                   - grade (str): The final grade ("Excellent", "Acceptable", "Failed").
                   - reward (float): The terminal reward associated with the final grade.
        """
        student = ThesisStudentSimulator(student_type)
        logs = []
        # Evolve the student's state for the specified trajectory length
        for _ in range(trajectory_length):
            logs.append(student.evolve_one_step())
        # Grade the final outcome based on the last state in the trajectory
        grade, terminal_reward = ThesisStudentSimulator.grade_final_outcome(logs[-1])
        return logs, grade, terminal_reward


## Part 8: Multi-Student Synthetic Cohort Generator (ThesisCohortSimulator)

The `ThesisCohortSimulator` class is designed to generate a dataset of simulated thesis trajectories for an entire cohort of diverse synthetic students. This aggregated dataset is crucial for the initial pre-training of the Reinforcement Learning ethics supervisor, providing a broad range of scenarios and behaviors.

**Purpose:**
To efficiently create a large, varied dataset of synthetic student interactions and outcomes. This dataset is used to train the RL agent to develop a generalizable ethical policy across different student types before any potential per-student fine-tuning.

**Key Components:**
- `STUDENT_TYPES`: A class attribute list defining the different types of students that can be simulated ("Conservative", "Aggressive", "Struggling", "Stable Performer").
- `generate_cohort_dataset(num_students=100, trajectory_length=30)`: A static method that is the primary function of this class. It generates the specified number of synthetic students, each with a randomly assigned type, runs a full simulation trajectory for each using the `ThesisStudentSimulator`, and collects all the resulting logs, final grades, and terminal rewards into a single dataset. It also prints a summary of the final grades distribution within the generated cohort.

**How to Use:**
- To generate a dataset for a cohort of 100 students with trajectories of 30 steps each, simply call the static method: `cohort_data = ThesisCohortSimulator.generate_cohort_dataset(num_students=100, trajectory_length=30)`. The returned `cohort_data` is a list where each element is a dictionary containing the student type, their full trajectory of logs, their final grade, and the terminal reward.

**Interaction with Other Components:**
- **Thesis Student Simulator:** The `ThesisCohortSimulator` relies heavily on the `ThesisStudentSimulator.generate_full_trajectory_with_grading` static method to produce individual student trajectories and outcomes.
- **Synthetic RL Pretrainer:** The `SyntheticRLPretrainer` class uses the `generate_cohort_dataset` method to obtain the large pool of synthetic logs required for pre-training the RL agent. It then flattens the trajectories from this dataset into a single list of logs for the training process.
- **RLTrainingLoop:** Although not directly interacted with by this class, the dataset generated here is ultimately fed into the `RLTrainingLoop` by the `SyntheticRLPretrainer`.

**Data Structures and Configuration:**
- The class uses the predefined `STUDENT_TYPES` list.
- The output is a list of dictionaries, each representing a simulated student with their complete `trajectory` (a list of log dictionaries), their `grade` (string), and `final_reward` (float).


In [229]:
# ===========================================================
# PART 8 — MULTI-STUDENT SYNTHETIC COHORT GENERATOR
# ===========================================================

class ThesisCohortSimulator:
    """
    Generates a dataset of simulated thesis trajectories for a cohort of students.

    This class uses the `ThesisStudentSimulator` to create trajectories for
    multiple students of different types, providing a dataset for training
    and evaluating the RL agent.
    """
    STUDENT_TYPES = ["Conservative", "Aggressive", "Struggling", "Stable Performer"]

    @staticmethod
    def generate_cohort_dataset(num_students=100, trajectory_length=30):
        """
        Generates a dataset of simulated thesis trajectories for a cohort.

        Args:
            num_students (int): The number of students to simulate. Defaults to 100.
            trajectory_length (int): The number of steps in each student's trajectory.
                                     Defaults to 30.

        Returns:
            list of dict: A list of dictionaries, where each dictionary represents
                          a student and contains their trajectory, final grade,
                          and terminal reward.
        """
        dataset = []
        grade_summary = {"Excellent": 0, "Acceptable": 0, "Failed": 0}
        # Generate trajectories for the specified number of students
        for _ in range(num_students):
            # Randomly select a student type
            student_type = random.choice(ThesisCohortSimulator.STUDENT_TYPES)
            # Generate a full trajectory and grade for the student
            logs, grade, terminal_reward = ThesisStudentSimulator.generate_full_trajectory_with_grading(
                student_type, trajectory_length)
            dataset.append({
                "student_type": student_type,
                "trajectory": logs,
                "grade": grade,
                "final_reward": terminal_reward
            })
            grade_summary[grade] += 1 # Count the grades for summary

        # Print a summary of the generated cohort grades
        print("Cohort Generation Complete:")
        for grade, count in grade_summary.items():
            print(f"  {grade}: {count} students")
        return dataset



## Part 9: Synthetic RL Pretraining Pipeline (SyntheticRLPretrainer)

The `SyntheticRLPretrainer` class orchestrates the process of pre-training the Reinforcement Learning ethics supervisor using a large synthetic dataset generated by the `ThesisCohortSimulator`. This step is typically done before applying the RL agent to real student data to provide it with an initial understanding of the environment and ethical considerations.

**Purpose:**
To automate the generation of a comprehensive synthetic dataset and use it to train the `SupervisorRL` agent, establishing a foundational policy for ethical guidance.

**Key Components:**
- `__init__(config, model_path="ppo_ethics_model")`: Initializes the pretrainer. It takes the RL configuration and the desired model path as input, and internally initializes an `RLTrainingLoop` instance, which in turn manages the `SupervisorRL` agent.
- `run_synthetic_pretraining(num_students=100, trajectory_length=30)`: The main method to trigger the pretraining process. It first calls the `ThesisCohortSimulator.generate_cohort_dataset` method to get the synthetic data, then flattens the trajectories from all students into a single list of logs, and finally passes this aggregated list of logs to the `RLTrainingLoop.run_training_day` method to train the RL agent.

**How to Use:**
- To run the synthetic pretraining with default settings (100 students, 30 steps per trajectory), after initializing the `RLTrainingLauncher` (which initializes the `SyntheticRLPretrainer`), you would call `launcher.run_synthetic_full_pretraining()`. If you are using the `SyntheticRLPretrainer` directly, you would initialize it with a config and then call `pretrainer.run_synthetic_pretraining(num_students=200, trajectory_length=40)` to specify different parameters.

**Interaction with Other Components:**
- **RLConfigManager:** The pretrainer is initialized with the configuration loaded by `RLConfigManager`, which is then passed down to the `RLTrainingLoop` and `SupervisorRL`.
- **ThesisCohortSimulator:** It directly calls the `ThesisCohortSimulator.generate_cohort_dataset` static method to obtain the synthetic training data.
- **RLTrainingLoop:** It uses an instance of `RLTrainingLoop` to handle the actual process of feeding logs to the `DataPreprocessor` and training the `SupervisorRL` agent on this data.
- **SupervisorRL:** The training of the PPO agent is managed by the `RLTrainingLoop` instance held within the pretrainer.

**Data Structures and Configuration:**
- It works with the list of dictionaries returned by the `ThesisCohortSimulator`, processing the `trajectory` lists within that dataset.
- The training parameters (like the number of students and trajectory length) are passed as arguments to the `run_synthetic_pretraining` method.


In [230]:
# ===========================================================
# PART 9 — SYNTHETIC RL PRETRAINING PIPELINE
# ===========================================================

class SyntheticRLPretrainer:
    """
    Manages the pretraining of the RL agent using synthetic thesis student data.

    This class uses the `ThesisCohortSimulator` to generate a large dataset
    of synthetic logs and then trains the RL agent (`SupervisorRL`)
    using this data via the `RLTrainingLoop`.
    """
    def __init__(self, config, model_path="ppo_ethics_model"):
        """
        Initialize the synthetic pretrainer.

        Args:
            config (dict): The loaded RL configuration.
            model_path (str): The path to the PPO model storage. Defaults to "ppo_ethics_model".
        """
        # Corrected: Instantiate RLConfigManager first, then load config
        config_manager = RLConfigManager()
        self.config = config_manager.load_config() # Load configuration
        # Initialize the training loop with the config and model path
        self.training_loop = RLTrainingLoop(self.config, model_path)

    def run_synthetic_pretraining(self, num_students=100, trajectory_length=30):
        """
        Runs the full synthetic pretraining pipeline.

        Generates a synthetic cohort dataset and trains the RL agent on the
        collected trajectories.

        Args:
            num_students (int): The number of synthetic students to generate data for.
                                Defaults to 100.
            trajectory_length (int): The length of each student's trajectory.
                                     Defaults to 30.
        """
        print("\nGenerating synthetic cohort dataset for pretraining...")
        # Generate the synthetic dataset
        dataset = ThesisCohortSimulator.generate_cohort_dataset(num_students, trajectory_length)
        # Flatten the trajectories from all students into a single list of logs
        all_logs = []
        for student in dataset:
            all_logs.extend(student["trajectory"])

        print(f"Total synthetic logs for PPO training: {len(all_logs)}")
        # Run the training loop on the collected synthetic logs
        self.training_loop.run_training_day(all_logs)

## Part 10: Final Launcher (RLTrainingLauncher)

The `RLTrainingLauncher` class serves as the main entry point and master system launcher for the entire thesis RL assistant training and configuration system. It brings together all the previously defined components and provides different modes of operation for development, simulation, and training with real or synthetic data.

**Purpose:**
To provide a single interface for initializing the RL system components, launching the developer dashboard, running simulated training, handling real data training, managing online incremental updates, and executing the full synthetic pretraining pipeline.

**Key Components:**
- `__init__()`: Initializes the launcher by loading the RL configuration using `RLConfigManager`, initializing the `DataPreprocessor`, creating an instance of the `RLTrainingLoop` (which includes the `SupervisorRL` agent), and initializing the `SyntheticRLPretrainer`.
- `launch_dashboard()`: Launches the Streamlit-based `DeveloperDashboard` for interactive configuration of the RL system.
- `run_simulated_training(days=3, batch_size=5)`: Runs a step-by-step simulation of training using a `MockSimulator` to generate small batches of logs over several simulated "days". This mode is useful for basic testing and debugging of the training loop.
- `run_real_training(real_logs)`: Takes a list of actual collected usage logs (`real_logs`) and feeds them into the `RLTrainingLoop` for training the RL agent on real-world data.
- `run_online_incremental_training(incremental_logs)`: Designed for online learning scenarios. It takes a list of newly collected `incremental_logs` and uses the `RLTrainingLoop` to fine-tune the existing RL model with this new data.
- `run_synthetic_full_pretraining(num_students=100, trajectory_length=30)`: Triggers the full synthetic pretraining pipeline by calling the `run_synthetic_pretraining` method of the `SyntheticRLPretrainer` instance. This generates a large synthetic dataset and trains the RL agent on it.

**How to Use:**
- Instantiate the launcher: `launcher = RLTrainingLauncher()`.
- Select a mode of operation based on user input or script logic:
    - `launcher.launch_dashboard()`: To start the configuration dashboard (requires Streamlit).
    - `launcher.run_simulated_training(days=5, batch_size=10)`: To run a short simulated training session.
    - `launcher.run_real_training(my_real_logs)`: To train with your collected real logs.
    - `launcher.run_online_incremental_training(new_logs)`: To perform incremental online updates with new data.
    - `launcher.run_synthetic_full_pretraining(num_students=200, trajectory_length=50)`: To run the comprehensive synthetic pretraining.
- The `if __name__ == "__main__":` block provides a command-line-like interface to select the mode when the script is run directly.

**Interaction with Other Components:**
- **RLConfigManager:** Used during initialization to load the system configuration.
- **DataPreprocessor:** An instance is held and used by the `RLTrainingLoop` for processing logs.
- **RLTrainingLoop:** An instance is held and used by the launcher to perform training with both simulated and real/incremental logs.
- **SyntheticRLPretrainer:** An instance is held and used to execute the full synthetic pretraining pipeline.
- **DeveloperDashboard:** An instance is created and launched when the 'dashboard' mode is selected.
- **MockSimulator:** An internal mock class used specifically by `run_simulated_training` to generate synthetic logs for that mode.

**Data Structures and Configuration:**
- Relies on the dictionary structure of the RL configuration loaded by `RLConfigManager`.
- Processes lists of log dictionaries, as generated by the simulators or collected from real usage.
- Uses numerical parameters (like `days`, `batch_size`, `num_students`, `trajectory_length`) to control the simulation and training processes.


In [231]:

# PART 10 — FINAL LAUNCHER
# ===========================================================

class RLTrainingLauncher:
    """
    The main entry point and master system launcher for the thesis RL assistant training.

    This class orchestrates the different training modes (dashboard, simulated,
    real data, online, synthetic full pretraining) and initializes the
    necessary components (`RLConfigManager`, `DataPreprocessor`,
    `RLTrainingLoop`, `SyntheticRLPretrainer`).
    """
    def __init__(self):
        """
        Initialize the launcher by loading configuration and components.
        """
        print("RLTrainingLauncher: Initializing...")
        # Load the RL configuration
        config_manager = RLConfigManager()
        self.config = config_manager.load_config()
        print("RLTrainingLauncher: Configuration loaded.")

        # Initialize the data preprocessor
        # The data preprocessor will now need to handle state variables from multiple modules,
        # but its `extract_state` method already iterates through the config's `state_variables`,
        # so it should work correctly as long as the log entries contain the necessary data.
        self.preprocessor = DataPreprocessor(self.config)
        print("RLTrainingLauncher: DataPreprocessor initialized.")

        # Initialize the RL training loop (this also initializes the RL agent)
        # The RLTrainingLoop initializes SupervisorRL, which now handles module instances.
        self.training_loop = RLTrainingLoop(self.config)
        print("RLTrainingLauncher: RLTrainingLoop initialized.")

        # Initialize the synthetic pretrainer
        # The SyntheticRLPretrainer also initializes RLTrainingLoop.
        self.pretrainer = SyntheticRLPretrainer(self.config)
        print("RLTrainingLauncher: SyntheticRLPretrainer initialized.")
        print("RLTrainingLauncher: Initialization complete.")


    def launch_dashboard(self):
        """
        Launch the Streamlit-based developer interface for configuring the RL system.
        """
        # DeveloperDashboard is assumed to be defined elsewhere or in a previous cell
        # from developer_dashboard import DeveloperDashboard # Uncomment if in separate file
        try:
            dashboard = DeveloperDashboard() # Initialize the DeveloperDashboard
            dashboard.launch() # Launch the dashboard
        except NameError:
            print("Error: DeveloperDashboard class not found. Make sure the cell defining it has been executed.")


    def run_simulated_training(self, days=3, batch_size=5):
        """
        Simulate RL model training using synthetic logs generated step-by-step.

        Args:
            days (int): Number of training days to simulate. Defaults to 3.
            batch_size (int): Number of logs to generate per day. Defaults to 5.
        """
        print("\nRunning Simulated Step-by-Step Training...")
        class MockSimulator:
            """
            A mock simulator to generate synthetic log batches for step-by-step training.
            This mock simulator now needs to generate logs that include all
            state variables defined in the config across all modules.
            """
            def __init__(self, config):
                 self.config = config
                 self.state_variables = config.get("state_variables", [])
                 # Initialize internal mock state based on config vars
                 for var in self.state_variables:
                      if var == "timestep":
                           setattr(self, var, 0)
                      else:
                           setattr(self, var, np.random.rand()) # Initialize randomly

            def generate_batch(self, batch_size):
                """
                Generates a batch of mock log entries.

                Args:
                    batch_size (int): The number of logs to generate in the batch.

                Returns:
                    list of dict: A list of mock log entries.
                """
                # print("Generating mock log batch...") # Suppress frequent prints
                mock_logs = []
                for _ in range(batch_size):
                    # Simulate state evolution (simple random walk for mock)
                    for var in self.state_variables:
                         if var != "timestep": # Timestep is handled by the environment
                             current_val = getattr(self, var)
                             # Apply small random change, clip to [0, 1]
                             updated_val = np.clip(current_val + np.random.normal(0, 0.05), 0.0, 1.0)
                             setattr(self, var, updated_val)


                    # Create a log entry with the current mock state values
                    log_entry = {}
                    for var in self.state_variables:
                         log_entry[var] = getattr(self, var)

                    # Add some simulated event flags for reward calculation (matching reward_config keys)
                    # These should ideally be correlated with state, but random for simplicity here
                    log_entry["fluency_improved"] = random.random() < 0.1
                    log_entry["trust_earned"] = random.random() < 0.05
                    log_entry["creativity_expressed"] = random.random() < 0.08
                    log_entry["autonomy_respected"] = random.random() < 0.15
                    log_entry["ai_dependency_violation"] = random.random() < log_entry.get("ai_usage", 0.0) * 0.2
                    log_entry["ethical_boundary_crossed"] = random.random() < log_entry.get("ethical_flags", 0.0) * 0.1
                    log_entry["deadline_panic_detected"] = random.random() < log_entry.get("deadline_ratio", 0.0) * 0.15 # Assuming deadline_ratio is a state var
                    log_entry["milestone_completed"] = random.random() < 0.03
                    log_entry["novel_but_safe"] = random.random() < log_entry.get("creativity_score", 0.0) * 0.1
                    log_entry["supervisor_disappointment"] = random.random() < (1.0 - log_entry.get("advisor_feedback", 0.0)) * 0.1

                    # Add other potential log fields expected by the preprocessor or environment
                    log_entry["prompt"] = "simulated prompt"
                    log_entry["intent"] = "simulated intent"
                    log_entry["thesis_stage"] = "simulated stage"


                    mock_logs.append(log_entry)
                return mock_logs

        # Initialize the mock simulator with the loaded config
        self.simulator = MockSimulator(self.config)

        # Run simulation for the specified number of days
        for day in range(days):
            print(f"\nSimulated Day {day + 1}")
            logs = self.simulator.generate_batch(batch_size=batch_size)
            # Run the training loop on the generated batch of logs
            # The training loop uses the DataPreprocessor to convert these logs
            # into states and rewards for the SupervisorRL agent.
            self.training_loop.run_training_day(logs)

    def run_real_training(self, real_logs):
        """
        Train the RL model using real usage logs.

        Args:
            real_logs (list of dict): Collected real usage logs to use in training.
        """
        print("\nTraining with Real Logs...")
        # Run the training loop on the provided real logs
        self.training_loop.run_training_day(real_logs)

    def run_online_incremental_training(self, incremental_logs):
        """
        Run online incremental updates using newly gathered data.

        This method simulates receiving new logs incrementally and using them
        to fine-tune the already trained RL model.

        Args:
            incremental_logs (list of dict): New usage logs collected for fine-tuning.
        """
        print("\nIncremental Online Training...")
        # Run the training loop on the incremental logs for fine-tuning
        self.training_loop.run_training_day(incremental_logs)

    def run_synthetic_full_pretraining(self, num_students=100, trajectory_length=30):
        """
        Runs the full synthetic pretraining pipeline using a cohort simulator.

        Args:
            num_students (int): The number of synthetic students to generate data for.
                                Defaults to 100.
            trajectory_length (int): The length of each student's trajectory.
                                     Defaults to 30.
        """
        print("\nRunning Full Synthetic PPO Pretraining...")
        # Use the pretrainer component to run the synthetic pretraining
        # The pretrainer uses the ThesisCohortSimulator, which generates logs
        # that should contain all state variables defined in the config.
        self.pretrainer.run_synthetic_pretraining(num_students, trajectory_length)


if __name__ == "__main__":
    launcher = RLTrainingLauncher() # Initialize the main launcher
    print("RL Training System Entry Point")
    print("Modes: [dashboard] [train_simulated] [train_real] [train_online] [train_synthetic_full]")
    # Use a default mode for automated execution, or keep input() for interactive use
    mode = "train_simulated" # Set a default mode for testing
    # mode = input("Mode: ").strip() # Uncomment for interactive mode

    # Execute the selected mode
    if mode == "dashboard":
         # DeveloperDashboard is assumed to be defined elsewhere or in a previous cell
         # from developer_dashboard import DeveloperDashboard # Uncomment if in separate file
         try:
            dashboard = DeveloperDashboard() # Initialize the DeveloperDashboard
            dashboard.launch() # Launch the dashboard
         except NameError:
            print("Error: DeveloperDashboard class not found. Make sure the cell defining it has been executed.")
    elif mode == "train_simulated":
        launcher.run_simulated_training()
    elif mode == "train_real":
        print("Load your real usage logs into 'real_logs' and call launcher.run_real_training(real_logs)")
        # Example usage (commented out):
        # real_logs = [...] # Load your real logs here
        # launcher.run_real_training(real_logs)
    elif mode == "train_online":
        print("Load new incremental logs into 'incremental_logs' and call launcher.run_online_incremental_training(incremental_logs)")
        # Example usage (commented out):
        # incremental_logs = [...] # Load your new logs here
        # launcher.run_online_incremental_training(incremental_logs)
    elif mode == "train_synthetic_full":
        launcher.run_synthetic_full_pretraining()
    else:
        print("Invalid mode selected.")

RLTrainingLauncher: Initializing...
No config source provided, using default configuration.
Using default configuration.
✅ Configuration validated and loaded successfully (retaining Pydantic instances).
RLTrainingLauncher: Configuration loaded.
RLTrainingLauncher: DataPreprocessor initialized.
No config source provided, using default configuration.
Using default configuration.
✅ Configuration validated and loaded successfully (retaining Pydantic instances).


AttributeError: 'RewardItem' object has no attribute 'get'

In [233]:
# Re-run the RLTrainingLauncher initialization to test the fix
# This will also run the default simulated training mode if __name__ == "__main__"
launcher = RLTrainingLauncher()

RLTrainingLauncher: Initializing...
No config source provided, using default configuration.
Using default configuration.
✅ Configuration validated and loaded successfully (retaining Pydantic instances).
RLTrainingLauncher: Configuration loaded.
RLTrainingLauncher: DataPreprocessor initialized.
No config source provided, using default configuration.
Using default configuration.
✅ Configuration validated and loaded successfully (retaining Pydantic instances).
SupervisorRL: Initialized mock module instances: ['ethics', 'writing', 'emotion', 'idea']

--- SupervisorEnv Initialization Debug ---
Received config keys: ['config_version', 'created_at', 'state_variables', 'actions', 'ethics_threshold', 'action_effects', 'reward_config', 'action_transitions']
Config type: <class 'dict'>
Type of config['action_effects']: <class 'dict'>
Sample action_effects keys: ['eth_0', 'eth_1', 'eth_2', 'eth_3', 'brain_0']
Type of sample action_effects value ('eth_0'): <class '__main__.ActionEffects'>
Sample ac

In [235]:
# Re-run the RLTrainingLauncher initialization and simulated training
# This will execute the __main__ block of RLTrainingLauncher which
# is currently set to run the simulated training mode.
launcher = RLTrainingLauncher()

RLTrainingLauncher: Initializing...
No config source provided, using default configuration.
Using default configuration.
✅ Configuration validated and loaded successfully (retaining Pydantic instances).
RLTrainingLauncher: Configuration loaded.
RLTrainingLauncher: DataPreprocessor initialized.
No config source provided, using default configuration.
Using default configuration.
✅ Configuration validated and loaded successfully (retaining Pydantic instances).
SupervisorRL: Initialized mock module instances: ['ethics', 'writing', 'emotion', 'idea']

--- SupervisorEnv Initialization Debug ---
Received config keys: ['config_version', 'created_at', 'state_variables', 'actions', 'ethics_threshold', 'action_effects', 'reward_config', 'action_transitions']
Config type: <class 'dict'>
Type of config['action_effects']: <class 'dict'>
Sample action_effects keys: ['eth_0', 'eth_1', 'eth_2', 'eth_3', 'brain_0']
Type of sample action_effects value ('eth_0'): <class '__main__.ActionEffects'>
Sample ac

In [236]:
# Run the full synthetic pretraining pipeline
# This will generate a synthetic cohort and train the PPO agent on the data.
print("\n--- Running Full Synthetic Pretraining Pipeline ---")
launcher.run_synthetic_full_pretraining(num_students=50, trajectory_length=30)
print("\n--- Full Synthetic Pretraining Pipeline Complete ---")


--- Running Full Synthetic Pretraining Pipeline ---

Running Full Synthetic PPO Pretraining...

Generating synthetic cohort dataset for pretraining...
Cohort Generation Complete:
  Excellent: 21 students
  Acceptable: 9 students
  Failed: 20 students
Total synthetic logs for PPO training: 1500
Processing batch of 1500 logs for training...
Processed Log → State: [0.21267453 0.32656255 0.03791151 0.61089444 0.03       0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.        ], Reward: 0.0
Processed Log → State: [0.21886961 0.37035507 0.03024677 0.6208464  0.06       0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.        ], Reward: 0.0
Processed Log → State: [0.1935264  0.3890167  0.01910003 0.67522067 0.09       0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.        ], Reward: 0.0
Processed Log → State: [0.21406217 0.38257986 0.02006984 0

In [237]:
# Test the LangGraph with the integrated SupervisorRL and multi-module environment

print("\n--- Testing LangGraph with SupervisorRL and Multi-Module Env ---")

# 1. Load configuration using the RLConfigManager
try:
    mgr = RLConfigManager()
    config = mgr.load_config()
    print("✅ Configuration loaded.")

    # Ensure action_transitions exist for the demo, add if not
    # These are needed for the graph structure and the apply_action_effects decorator lookup
    if "action_transitions" not in config:
         config["action_transitions"] = {
             "eth_0": {"next": "write_0"},
             "write_0": {"next": "emo_0"},
             "emo_0": {"next": "brain_0"},
             "brain_0": {"next": "END"}
         }
         print("ℹ️ Added default action_transitions for demo.")


    # Ensure state_variables are correctly populated from action effects for the demo
    # This is important for the environment's observation space and the state vector
    vars_from_effects = set()
    if "action_effects" in config:
        for effects in config["action_effects"].values():
            vars_from_effects.update(effects.keys())
    required_state_vars = set(config.get("state_variables", []))
    missing_vars_in_state_vars = list(vars_from_effects - required_state_vars - set(["timestep"]))
    if missing_vars_in_state_vars:
        print(f"  Adding missing variables from action_effects to state_variables in config for demo: {missing_vars_in_state_vars}")
        config.setdefault("state_variables", []).extend(missing_vars_in_state_vars)
        config["state_variables"] = list(dict.fromkeys(config["state_variables"])) # Ensure uniqueness


except Exception as e:
    print(f"❌ Failed to load configuration: {e}")
    # Fallback to a basic default config if loading fails
    config = {
        "actions": {
            "dummy_action_1": "A first dummy action",
            "dummy_action_2": "A second dummy action",
        },
        "action_effects": {
            "dummy_action_1": {"timestep": 0.1}, # Add a simple effect
            "dummy_action_2": {"timestep": 0.1}, # Add a simple effect
        },
        "action_transitions": {
            "dummy_action_1": {"next": "dummy_action_2"},
            "dummy_action_2": {"next": "END"},
        },
        "state_variables": ["timestep"], # Minimal state variables
         "reward_config": {},
    }
    print("⚠️ Using fallback configuration.")


# 2. Initialize mock module instances (required by SupervisorEnv and SupervisorRL)
# These instances will hold the state variables
mock_module_instances = {
    "ethics": SimpleMockEthicsModule(config),
    "writing": SimpleMockWrittingModule(config),
    "emotion": SimpleMockEmotionModule(config),
    "idea": SimpleMockIdeaModule(config),
}
print("\nInitialized mock module instances for LangGraph test:", list(mock_module_instances.keys()))

# 3. Initialize the SupervisorRL agent
# This also initializes the SupervisorEnv internally with the mock module instances
try:
    # Make rl_supervisor globally accessible for the run_rl_policy node (as done in the main block)
    global rl_supervisor
    rl_supervisor = SupervisorRL(config)
    print("✅ SupervisorRL initialized for LangGraph test.")
except NameError:
    print("❌ SupervisorRL class not found. Cannot run test.")
    rl_supervisor = None


# 4. Define the initial state for the simulation
# The core state will be populated by the environment's reset method
initial_state: ThesisState = {
    "core": {}, # Start with an empty core state, will be populated by env.reset()
    "policy_trace": [], # Start with an empty trace
    "log": [], # Start with an empty log
    "config": config, # Pass the loaded config to the state
    "env": None, # Env will be initialized by SupervisorRL, need to get it from there
    "done": False, # Initial episode status
    "truncated": False, # Initial episode status
    # No need for "next_node" in the initial state, as the entry point is set directly
}

# 5. Get the environment instance from the initialized SupervisorRL and reset it
env = None
if rl_supervisor:
     env = rl_supervisor.env
     initial_state["env"] = env

     # Reset the environment to get the initial observation and populate the core state
     print("\nResetting environment before running LangGraph...")
     initial_obs, initial_info = env.reset()
     # Populate the initial core state from the environment's reset output
     state_variables_order = config.get("state_variables", [])
     initial_core_state = {}
     # Ensure the number of variables in config matches the observation space from env.reset()
     if len(state_variables_order) != len(initial_obs):
         print(f"Error: Number of state variables in config ({len(state_variables_order)}) does not match observation space size from env.reset() ({len(initial_obs)}).")
         # Attempt to proceed but log the error
         # Use min length to avoid index errors
         num_vars_to_map = min(len(state_variables_order), len(initial_obs))
         for i in range(num_vars_to_map):
             initial_core_state[state_variables_order[i]] = float(initial_obs[i])
     else:
         for i, var_name in enumerate(state_variables_order):
              initial_core_state[var_name] = float(initial_obs[i]) # Convert numpy float to standard float


     initial_state["core"] = initial_core_state
     initial_state["done"] = False # Ensure done flag is False after reset
     initial_state["truncated"] = False # Ensure truncated flag is False after reset


     print("Initial core state populated from env.reset():", dict(list(initial_state["core"].items())[:5])) # Print sample
     print("Initial env done flag:", initial_state["done"])
     print("Initial env truncated flag:", initial_state["truncated"])
     print("Initial env timestep:", env.timestep) # Should be 0 after reset
else:
    print("Cannot run LangGraph test without a valid SupervisorRL.")


# 6. Build the LangGraph with the RL policy node
thesis_graph = None
if rl_supervisor and env: # Only proceed if SupervisorRL and env were initialized successfully
    try:
        # Use the new build function that includes the RL node
        thesis_graph = build_policy_graph_with_rl(config)
        print("\n✅ LangGraph built with RL policy node.")

    except Exception as e:
        print(f"\n❌ An error occurred during LangGraph building: {e}")


# 7. Run the graph for a few steps using stream
if thesis_graph:
    print("\nRunning LangGraph stream for a few steps...")
    # The graph will start at the entry point ('rl_policy_node')
    # The graph will continue until the RL policy node returns "END"
    # or the environment signals done/truncated via the apply_action_effects decorator
    # The max_steps argument limits the number of steps in the graph traversal,
    # preventing infinite loops in case of unexpected routing.
    # Set a reasonable limit for the demo to observe a few action/policy cycles.
    max_graph_steps = 10 # Limit the number of graph steps for the test

    try:
        # Stream the graph execution
        for step_output in thesis_graph.stream(initial_state, {"recursion_limit": max_graph_steps}):
             # The stream method yields the state after each node execution.
             # This allows us to see the state changes step-by-step.
             for key, value in step_output.items():
                 print(f"\n--- State after '{key}' node ---")
                 # Print relevant parts of the state
                 print("Core (sample):", dict(list(value.get("core", {}).items())[:5])) # Print sample key-value pairs
                 print("Timestep:", value.get("core", {}).get("timestep", "N/A"))
                 print("Policy Trace:", value.get("policy_trace", [])[-5:]) # Last few trace items
                 print("Log (last entry):", value.get("log", [])[-1] if value.get("log") else "Empty")
                 print("Done:", value.get("done", False))
                 print("Truncated:", value.get("truncated", False))
                 # Check the next_node if it exists (set by RL node)
                 if "next_node" in value:
                     print("Next Node (set by RL node):", value["next_node"])

                 # Check for episode termination after each step
                 if value.get("done", False) or value.get("truncated", False):
                      print("\nEpisode terminated during stream.")
                      break # Exit the loop if the episode is done


        print("\n✅ LangGraph stream finished (complete or limit reached).")

    except Exception as e:
        print(f"\n❌ An error occurred during LangGraph stream execution: {e}")


print("\n--- LangGraph Test Complete ---")


--- Testing LangGraph with SupervisorRL and Multi-Module Env ---
No config source provided, using default configuration.
Using default configuration.
✅ Configuration validated and loaded successfully (retaining Pydantic instances).
✅ Configuration loaded.
❌ Failed to load configuration: 'ActionEffects' object has no attribute 'keys'
⚠️ Using fallback configuration.

Initialized mock module instances for LangGraph test: ['ethics', 'writing', 'emotion', 'idea']
SupervisorRL: Initialized mock module instances: ['ethics', 'writing', 'emotion', 'idea']

--- SupervisorEnv Initialization Debug ---
Received config keys: ['actions', 'action_effects', 'action_transitions', 'state_variables', 'reward_config']
Config type: <class 'dict'>
Type of config['action_effects']: <class 'dict'>
Sample action_effects keys: ['dummy_action_1', 'dummy_action_2']
Type of sample action_effects value ('dummy_action_1'): <class 'dict'>
Sample action_effects value: {'timestep': 0.1}
Type of config['reward_config']